# 19 — LP-relaksasjon av hele vassdraget som input til v5-boosting

**Henrik Remmen (541747), Gruppe 119 — [119] SINTEF Databreach**

Beste public-score nå: `multiscale_boosting_v5.csv` **0.99036**. Rapportert leder: 0.99562.
Denne notebooken er ett nytt, kontrollert forsøk. Den lover ingen score.

## Hypotese (presis)

*H1:* Å legge til per-stasjon-features fra en ukentlig **lineær relaksasjon (LP) av hele
Tokke–Vinje-nettet** (planlagt vannføring, redusert-kost-margin, dual-basert terskel og
planlagt magasinbane) til **nøyaktig** v5-modellen øker renset (purged) forward micro-ROC-AUC.
Forventet mekanisme (del av hypotesen, ikke vist): færre feil i ukentlig driftsnivå for
de hydraulisk koblede enhetene (Vinje, Tokke, Hogga, Lio, Haukeli). Alt annet er likt: samme 334
v5-features, XGBoost/LightGBM-parametere, seeds, 2× arvede trærantall og 50/50-blanding.

## Lokalt grunnlag (analyser gjort før notebooken ble bygd, uten ny boosting)

- **Nivåfeil er viktige, men ikke hele bildet.** Et label-informert orakel som kjenner hver
  case/enhets sanne ukentlige ON-andel gir AUC 0.9974/0.9969 (2021/2022) mot v5 0.9851/0.9868.
  Det er et tak, ikke bevis på riktig timing: orakelet får alle konstante case/enheter gratis
  (47 % i 2021, 39 % i 2022). Innen uker med både ON og OFF er v5s AUC 0.958/0.968 (vektet
  per positiv/negativ-par), så timingfeil betyr fortsatt noe.
- **Regelen pris > vannverdi** (kildemagasinets lokale EUR/MWh) er nesten perfekt for Byrte,
  Songa, Kjela og Vesle Kjela, men svak for koblede enheter (snitt 2019–22): Vinje G1 0.756,
  Haukeli 0.774, Lio 0.871, Tokke 0.915–0.929, Hogga 0.934.
- **Én utrent LP-feature** gir samme mål: Vinje G1 0.978, Haukeli 0.933, Lio 0.949,
  Tokke 0.981–0.986, Hogga 0.968. I 2022 er LP alene på høyde med v5 for Tokke G1/Vinje G1.
- **Lio** kjører under vannverdi nesten bare når minstevannføringen i r_Vest_Vassdraget ikke
  dekkes av naturlig tilsig (ON 43–93 % vs 1–28 %); LP-en fanger dette via dualverdien.
- **LP alene løser ikke nivået:** ukenivå-MAE 2022 er lavere enn v5 for bare 8 av 14 enheter,
  og klart høyere for Lio (0.194 vs 0.114) og Vinje G3 (0.104 vs 0.051). **Som tillegg hjelper
  den:** en enkel lineær nivåmodell tilpasset 2021 og målt 2022 går fra MAE 0.0576 (bare
  v5-nivå) til 0.0490 (v5 + LP-nivå), bedre for 12 av 14 enheter (ikke Lio og Songa).
- Liten stacker på lagrede out-of-sample-prediksjoner (tilpasset 2021, målt 2022):
  kontroll uten LP 0.97821, med LP 0.98612 (+0.0079). Én-års-stacking skader kalibrering
  mellom enheter (begge under v5 0.98682), så **den reelle testen er denne notebooken**.

## Validering og kontroll

- Utvidende tidsvinduer med purge: trening kun på case der hele 168-t-horisonten slutter før
  året starter. Tilstøtende daglige case deler 144 timer.
- **Modellvalg:** 2020 og 2021. Forhåndsregistrert regel: blend-delta i snitt > 0 og intet år
  dårligere enn −0.0005. Ingen andre valg: ingen vektsøk, ingen trærantallvalg
  (1×/1.5× lagres kun som diagnose).
- **Etterfølgende kontroll:** 2022, kjøres én gang etter at `selection.json` finnes.
- **Ærlighet:** 2020–2022 er brukt i tidligere arbeid, og 2022 ble brukt i proben over.
  Ingen av dem er urørte holdout-år. Blokk-bootstrap (7-dagers blokker) gir usikkerhet.
- **Konklusjon før slutttrening**, bare fra resultater med samme data/kode/parametere:
  `SUPPORTED` (valgt og 90 %-intervall i kontrollåret over 0), `POSITIVE BUT UNCERTAIN`
  (positivt punktestimat, intervallet når 0), `NOT CONFIRMED` eller `NOT SUPPORTED`.
  Sluttcellen nekter å kjøre mot valideringer fra et annet oppsett og skriver
  `final_manifest.json` som knytter CSV-en til verdikten. `TRAIN_FINAL = False` stopper før
  de 28 slutt-fitene.
- Logges: micro-AUC per år/modell/trærprefiks, AUC/Brier per enhet, startmåned,
  horisontblokk og UTC-time, ukenivå-MAE, kjøretid per enhet (`runtime_log.csv`).

## Ikke brukt

Ingen eksterne data, skjulte testetiketter, observerte volumer etter case-start, SHOP-resultater,
`total_calculation_time` eller Run No. LP-en er vår egen funksjon av tillatte inputs
(pris, tilsig, startvolum, sluttvannverdi, min-vannføring/-volum, topologi).

## Kjøring og ressurser

Kjør alle celler i en fersk kernel. Første kodecelle styrer tråder og år.
Fullførte enheter gjenopptas fra disk; en avbrutt enkelttrening starter på nytt.
Notebook-18-logger (8 tråder, 2× trær): ett 14-enheters sett tar ca. 9–11 min (XGB) og 12–16 min (LGB).
Utvikling (2 år × 4 sett) + kontroll (4 sett) + final (2 sett) ≈ 14 sett ≈ **2,5–3 t**.
LP for 3 647 case: ca. 1–3 min. Kortnotebooken (4 tråder, 28 fits) er anslått til 1–2 t, **ikke målt**.

In [1]:
# FIRST CODE CELL: execution resources and experiment years. Statistical settings live in config.
import os
RUN_PROFILE = 'local'      # 'professor': max 4 CPU threads and no cache reads (cold run)
LOCAL_CPU_THREADS = 8      # 8 matches the thread count of the recorded notebook-18 runs
DEV_YEARS = (2020, 2021)   # model selection (pre-registered rule, see markdown above)
AUDIT_YEAR = 2022          # later check, evaluated once after selection.json exists
TRAIN_FINAL = False         # False: stop after the printed verdict, before the 28 final fits
if RUN_PROFILE not in ('local', 'professor'):
    raise ValueError('Unknown profile')
THREADS = min(LOCAL_CPU_THREADS if RUN_PROFILE == 'local' else 4, os.cpu_count() or 1)
USE_CHECKPOINTS = RUN_PROFILE == 'local'
for key in ('OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[key] = '1'
os.environ['OMP_NUM_THREADS'] = str(THREADS)
print(f'{RUN_PROFILE}: {THREADS} CPU threads, one model at a time, resume={USE_CHECKPOINTS}')

local: 8 CPU threads, one model at a time, resume=True


In [2]:
from pathlib import Path
import json
import time
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_STARTED = time.perf_counter()
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/kernel').is_dir() and (p / 'prediction_mapping.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Place the notebook beside the supplied data/ directory and mapping files.')
SOURCE_DIGEST = '4c8816d61b4d2c1820d83994c04466d5a5bdc4b169b04529052a450559e683a9'  # change if you edit any definition cell (invalidates caches)
ARTIFACTS = ROOT / 'artifacts/hydro_lp_v6'

# Identical to multiscale_boosting_v5 (Kaggle public 0.99036): inherited per-unit counts x 2.
BASELINE_ROUNDS = {
 'xgb': dict(zip(
  ['Hogga_G1','Tokke_G1','Tokke_G2','Tokke_G3','Tokke_G4','Lio_G1','Byrte_G1',
   'Vinje_G1','Vinje_G2','Vinje_G3','Haukeli_G1','Songa_G1','Kjela_G1','Vesle Kjela_G1'],
  [410,450,854,587,693,249,292,658,404,607,223,510,326,372])),
 'lgb': dict(zip(
  ['Hogga_G1','Tokke_G1','Tokke_G2','Tokke_G3','Tokke_G4','Lio_G1','Byrte_G1',
   'Vinje_G1','Vinje_G2','Vinje_G3','Haukeli_G1','Songa_G1','Kjela_G1','Vesle Kjela_G1'],
  [398,534,387,458,410,386,441,852,299,873,330,465,393,177]))}
xgb_parameters = dict(objective='binary:logistic', eval_metric='auc', learning_rate=.03,
    max_depth=7, min_child_weight=50, subsample=.85, colsample_bytree=.85,
    reg_alpha=.1, reg_lambda=5., tree_method='hist', max_bin=256, device='cpu', random_state=42)
lgb_parameters = dict(objective='binary', metric='None', learning_rate=.03, num_leaves=31,
    max_depth=-1, min_data_in_leaf=150, feature_fraction=.85, bagging_fraction=.85,
    bagging_freq=1, lambda_l1=.1, lambda_l2=5., max_bin=255, device_type='cpu',
    deterministic=True, force_col_wise=True, seed=42, verbosity=-1)
ROUNDS = BASELINE_ROUNDS
SPECS = {
    'xgb': dict(backend='xgboost', features='competition', max_rounds=2000, parameters=xgb_parameters),
    'lgb': dict(backend='lightgbm', features='competition', max_rounds=1800, parameters=lgb_parameters),
}

## pipeline — definitions only

In [3]:
"""Shared, target-safe data and feature pipeline used by experiment notebooks.

The module contains no model training.  It exists so CatBoost and XGBoost
experiments use exactly the same inputs, folds, metric, and submission layout.
"""

from dataclasses import dataclass
from pathlib import Path
import re

import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score


HORIZON = 168
RANDOM_SEED = 42


@dataclass
class ProjectData:
    root: Path
    X_all: pd.DataFrame
    target_cube: np.ndarray
    units: list[str]
    train_cases: pd.DataFrame
    test_cases: pd.DataFrame
    sample_submission: pd.DataFrame
    feature_sets: dict[str, list[str]]
    feature_group: dict[str, str]
    generator_features: pd.DataFrame
    folds: list[dict]

    @property
    def n_train_cases(self):
        return len(self.train_cases)

    @property
    def n_test_cases(self):
        return len(self.test_cases)

    @property
    def n_all_cases(self):
        return self.n_train_cases + self.n_test_cases


def read_time_csv(path, time_column="date"):
    frame = pd.read_csv(path)
    frame[time_column] = pd.to_datetime(frame[time_column], utc=True)
    return frame


def rows_for_cases(case_ids):
    case_ids = np.asarray(case_ids)
    return (
        case_ids[:, None] * HORIZON
        + np.arange(HORIZON)[None, :]
    ).ravel()


def micro_auc(y_true_cube, y_pred_cube):
    if y_true_cube.shape != y_pred_cube.shape:
        raise ValueError("Truth and prediction cubes must have identical shapes")
    return roc_auc_score(
        y_true_cube.reshape(len(y_true_cube), -1),
        y_pred_cube.reshape(len(y_pred_cube), -1),
        average="micro",
    )


def per_unit_auc(y_true_cube, y_pred_cube, units):
    return {
        unit: roc_auc_score(
            y_true_cube[:, unit_index, :].ravel(),
            y_pred_cube[:, unit_index, :].ravel(),
        )
        for unit_index, unit in enumerate(units)
    }


def make_purged_folds(train_cases, validation_years=(2020, 2021, 2022)):
    starts = pd.DatetimeIndex(train_cases["starttime"])
    folds = []
    for year in validation_years:
        validation_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
        validation_end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")
        validation_cases = np.flatnonzero(
            (starts >= validation_start) & (starts < validation_end)
        )
        training_cases = np.flatnonzero(
            starts + pd.Timedelta(hours=HORIZON - 1) < validation_start
        )
        if not len(training_cases) or not len(validation_cases):
            raise ValueError(f"Empty training or validation partition for {year}")
        if not (
            starts[training_cases].max() + pd.Timedelta(hours=HORIZON - 1)
            < starts[validation_cases].min()
        ):
            raise AssertionError(f"Fold {year} is not temporally purged")
        folds.append({
            "year": year,
            "train_cases": training_cases,
            "validation_cases": validation_cases,
        })
    return folds


def make_submission(data, prediction_cube, filename):
    expected = (data.n_test_cases, len(data.units), HORIZON)
    if prediction_cube.shape != expected:
        raise ValueError(f"Expected predictions {expected}, got {prediction_cube.shape}")

    submission_data = {"Run No": data.test_cases["Run No"].to_numpy()}
    for unit_index, unit in enumerate(data.units):
        for horizon in range(HORIZON):
            submission_data[f"result_committed_{unit}_t{horizon}"] = (
                prediction_cube[:, unit_index, horizon]
            )

    submission = pd.DataFrame(submission_data)[data.sample_submission.columns]
    probabilities = submission.iloc[:, 1:].to_numpy()
    if not submission["Run No"].equals(data.sample_submission["Run No"]):
        raise AssertionError("Run No does not match sample_submission.csv")
    if not np.isfinite(probabilities).all():
        raise ValueError("Submission contains missing or non-finite predictions")
    if probabilities.min() < 0 or probabilities.max() > 1:
        raise ValueError("Submission probabilities are outside [0, 1]")

    path = data.root / "submissions" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(path, index=False)
    return path, submission


def build_project_data(root):
    root = Path(root).resolve()
    data_dir = root / "data"

    price = read_time_csv(data_dir / "kernel" / "Historical_day_ahead_price_2015_2025.csv")
    inflow = read_time_csv(data_dir / "kernel" / "Historical_inflow_1958_2025.csv")
    volume = read_time_csv(data_dir / "kernel" / "Historical_volume_2015_2024.csv")
    water_value = read_time_csv(data_dir / "kernel" / "Synthetic_water_value_2015_2024.csv")
    min_flow = read_time_csv(data_dir / "extended" / "Constraint_min_flow.csv")
    min_volume = read_time_csv(data_dir / "extended" / "Constraint_min_volume.csv")
    targets = read_time_csv(data_dir / "kernel" / "Unit_commitment_decisions.csv", "starttime")
    prediction_mapping = read_time_csv(root / "prediction_mapping.csv", "starttime")
    sample_submission = pd.read_csv(root / "sample_submission.csv")

    with open(data_dir / "extended" / "Tokke_Vinje_topology.yaml") as stream:
        topology = yaml.safe_load(stream)

    pattern = re.compile(r"^result_committed_(.+)_t(\d+)$")
    target_columns = [column for column in targets if pattern.match(column)]
    units = list(dict.fromkeys(pattern.match(column).group(1) for column in target_columns))
    if len(units) != 14 or len(target_columns) != 14 * HORIZON:
        raise AssertionError("Expected 14 units and 168 targets per unit")
    if target_columns != list(sample_submission.columns[1:]):
        raise AssertionError("Target order differs from sample submission")

    target_cube = np.stack([
        targets[[f"result_committed_{unit}_t{hour}" for hour in range(HORIZON)]]
        .to_numpy(dtype=np.int8)
        for unit in units
    ], axis=1)
    if not set(np.unique(target_cube)).issubset({0, 1}):
        raise AssertionError("Targets must be binary")

    train_cases = targets[["Run No", "starttime"]].copy()
    test_cases = prediction_mapping[["Run No", "starttime"]].copy()
    all_cases = pd.concat([train_cases, test_cases], ignore_index=True)
    n_all_cases = len(all_cases)
    starts = pd.DatetimeIndex(all_cases["starttime"])
    horizon_hour = np.tile(np.arange(HORIZON), n_all_cases)
    target_time = pd.DatetimeIndex(
        starts.repeat(HORIZON) + pd.to_timedelta(horizon_hour, unit="h")
    )

    def numeric_indexed(frame):
        return frame.set_index("date").select_dtypes(include=np.number)

    def hourly_matrix(series):
        values = series.reindex(target_time).to_numpy()
        matrix = values.reshape(n_all_cases, HORIZON)
        if not np.isfinite(matrix).all():
            raise ValueError(
                f"Missing/non-finite hourly inputs: {(~np.isfinite(matrix)).sum()}"
            )
        return matrix

    def repeat_case(values):
        values = np.asarray(values)
        if len(values) != n_all_cases:
            raise ValueError("Case-level array has incorrect length")
        return np.repeat(values, HORIZON)

    def cyclic(values, period):
        angle = 2 * np.pi * np.asarray(values) / period
        return np.sin(angle), np.cos(angle)

    def clipped_shift(matrix, offset):
        indices = np.clip(np.arange(HORIZON) + offset, 0, HORIZON - 1)
        return matrix[:, indices]

    def centred_rolling_mean(matrix, window):
        return (
            pd.DataFrame(matrix).T
            .rolling(window=window, min_periods=1, center=True)
            .mean().T.to_numpy()
        )

    feature_data = {}
    feature_group = {}

    def add_feature(name, values, group):
        values = np.asarray(values)
        expected = n_all_cases * HORIZON
        if values.ndim != 1 or len(values) != expected:
            raise ValueError(f"{name}: expected {expected}, got {values.shape}")
        feature_data[name] = values.astype(np.float32)
        feature_group[name] = group

    hour = target_time.hour.to_numpy()
    weekday = target_time.dayofweek.to_numpy()
    month = target_time.month.to_numpy()
    for name, values, period in (
        ("hour", hour, 24),
        ("weekday", weekday, 7),
        ("month", month - 1, 12),
    ):
        sine, cosine = cyclic(values, period)
        add_feature(f"{name}_sin", sine, "calendar")
        add_feature(f"{name}_cos", cosine, "calendar")
    add_feature("horizon_hour", horizon_hour, "calendar")

    price_matrix = hourly_matrix(
        price.set_index("date")["Day-ahead price (EUR/MWh)"]
    )
    price_mean = price_matrix.mean(axis=1)
    price_std = price_matrix.std(axis=1)
    price_rank = pd.DataFrame(price_matrix).rank(axis=1, pct=True).to_numpy()
    add_feature("price", price_matrix.ravel(), "price")
    add_feature("price_week_mean", repeat_case(price_mean), "price")
    add_feature("price_week_std", repeat_case(price_std), "price")
    add_feature("price_week_percentile", price_rank.ravel(), "price")
    add_feature("price_minus_week_mean", (price_matrix - price_mean[:, None]).ravel(), "price")
    add_feature(
        "price_week_zscore",
        ((price_matrix - price_mean[:, None]) / (price_std[:, None] + 1e-6)).ravel(),
        "price_windows",
    )
    add_feature("price_top_10pct", (price_rank >= 0.90).ravel(), "price_windows")
    add_feature("price_bottom_10pct", (price_rank <= 0.10).ravel(), "price_windows")
    for offset in (1, 3, 6, 24):
        lag = clipped_shift(price_matrix, -offset)
        lead = clipped_shift(price_matrix, offset)
        add_feature(f"price_lag_{offset}h", lag.ravel(), "price_windows")
        add_feature(f"price_lead_{offset}h", lead.ravel(), "price_windows")
        add_feature(f"price_change_from_{offset}h", (price_matrix - lag).ravel(), "price_windows")
        add_feature(f"price_change_to_next_{offset}h", (lead - price_matrix).ravel(), "price_windows")
    for window in (3, 6, 12, 24, 48):
        rolling = centred_rolling_mean(price_matrix, window)
        add_feature(f"price_mean_{window}h", rolling.ravel(), "price_windows")
        add_feature(f"price_minus_mean_{window}h", (price_matrix - rolling).ravel(), "price_windows")

    reservoirs = [column for column in volume.columns if column != "date"]
    capacity = {
        name: values["max_vol"]
        for name, values in topology["model"]["reservoir"].items()
    }
    start_volume = numeric_indexed(volume).reindex(starts)[reservoirs]
    start_water = numeric_indexed(water_value).reindex(starts)[reservoirs]
    end_water = numeric_indexed(water_value).reindex(
        starts + pd.Timedelta(days=7)
    )[reservoirs]
    if start_volume.isna().any().any() or start_water.isna().any().any() or end_water.isna().any().any():
        raise ValueError("Missing case-level reservoir data")

    for reservoir in reservoirs:
        fill = start_volume[reservoir].to_numpy() / capacity[reservoir]
        water_start = start_water[reservoir].to_numpy()
        water_end = end_water[reservoir].to_numpy()
        add_feature(f"fill__{reservoir}", repeat_case(fill), "hydrology")
        add_feature(f"water_value_end__{reservoir}", repeat_case(water_end), "water_margins")
        add_feature(f"water_value_change__{reservoir}", repeat_case(water_end - water_start), "water_margins")
        add_feature(f"price_margin__{reservoir}", (price_matrix - water_end[:, None]).ravel(), "water_margins")

    system_fill = (
        start_volume.sum(axis=1).to_numpy()
        / sum(capacity[reservoir] for reservoir in reservoirs)
    )
    water_end_mean = end_water.mean(axis=1).to_numpy()
    add_feature("system_filling_ratio", repeat_case(system_fill), "hydrology")
    add_feature("water_value_end_mean", repeat_case(water_end_mean), "water_margins")
    margin = price_matrix - water_end_mean[:, None]
    add_feature("price_minus_water_value_mean", margin.ravel(), "water_margins")
    add_feature(
        "price_to_abs_water_value_ratio",
        (price_matrix / (np.abs(water_end_mean[:, None]) + 1.0)).ravel(),
        "water_margins",
    )

    inflow_values = numeric_indexed(inflow)
    inflow_matrices = {}
    for column in inflow_values:
        matrix = hourly_matrix(inflow_values[column])
        inflow_matrices[column] = matrix
        add_feature(f"inflow__{column}", matrix.ravel(), "hydrology")
        add_feature(f"inflow_week_mean__{column}", repeat_case(matrix.mean(axis=1)), "hydrology")
    total_inflow = np.sum(list(inflow_matrices.values()), axis=0)
    add_feature("inflow_total", total_inflow.ravel(), "hydrology")
    add_feature("inflow_total_week_mean", repeat_case(total_inflow.mean(axis=1)), "hydrology")
    add_feature("inflow_total_week_std", repeat_case(total_inflow.std(axis=1)), "hydrology")
    add_feature("inflow_total_24h_mean", centred_rolling_mean(total_inflow, 24).ravel(), "hydrology")
    add_feature(
        "inflow_total_cumulative_mean",
        (np.cumsum(total_inflow, axis=1) / (np.arange(HORIZON)[None, :] + 1)).ravel(),
        "hydrology",
    )

    flow_values = numeric_indexed(min_flow)
    min_volume_values = numeric_indexed(min_volume)
    flow_matrices = {}
    min_volume_matrices = {}
    for column in flow_values:
        matrix = hourly_matrix(flow_values[column])
        flow_matrices[column] = matrix
        add_feature(f"min_flow__{column}", matrix.ravel(), "constraints")
    for column in min_volume_values:
        matrix = hourly_matrix(min_volume_values[column])
        min_volume_matrices[column] = matrix
        add_feature(f"min_volume__{column}", matrix.ravel(), "constraints")
        if column in reservoirs:
            initial = start_volume[column].to_numpy()[:, None]
            add_feature(
                f"min_volume_slack__{column}",
                ((initial - matrix) / capacity[column]).ravel(),
                "constraints",
            )

    flow_active_count = np.sum([matrix > 0 for matrix in flow_matrices.values()], axis=0)
    volume_active_count = np.sum([matrix > 0 for matrix in min_volume_matrices.values()], axis=0)
    total_min_flow = np.sum(list(flow_matrices.values()), axis=0)
    total_min_volume = np.sum(list(min_volume_matrices.values()), axis=0)
    add_feature("min_flow_active_count", flow_active_count.ravel(), "constraints")
    add_feature("min_volume_active_count", volume_active_count.ravel(), "constraints")
    add_feature("min_flow_total", total_min_flow.ravel(), "constraints")
    add_feature("min_volume_total", total_min_volume.ravel(), "constraints")

    variable_flow_columns = [
        column for column in flow_values
        if 0 < (flow_values[column] > 0).mean() < 1
    ]
    variable_volume_columns = [
        column for column in min_volume_values
        if 0 < (min_volume_values[column] > 0).mean() < 1
    ]
    variable_active = np.zeros((n_all_cases, HORIZON), dtype=bool)
    for column in variable_flow_columns:
        variable_active |= flow_matrices[column] > 0
    for column in variable_volume_columns:
        variable_active |= min_volume_matrices[column] > 0
    add_feature("variable_constraint_active", variable_active.ravel(), "constraints")

    add_feature("margin_x_system_fill", (margin * system_fill[:, None]).ravel(), "interactions")
    add_feature("price_rank_x_system_fill", (price_rank * system_fill[:, None]).ravel(), "interactions")
    add_feature("margin_x_total_inflow", (margin * total_inflow).ravel(), "interactions")
    add_feature("price_rank_x_hour_sin", price_rank.ravel() * feature_data["hour_sin"], "interactions")
    add_feature("price_rank_x_hour_cos", price_rank.ravel() * feature_data["hour_cos"], "interactions")
    add_feature("margin_x_constraint", margin.ravel() * variable_active.ravel(), "interactions")

    X_all = pd.DataFrame(feature_data)
    constants = [
        column for column in X_all
        if X_all[column].nunique(dropna=False) <= 1
    ]
    X_all = X_all.drop(columns=constants)
    if X_all.isna().any().any() or not np.isfinite(X_all.to_numpy()).all():
        raise ValueError("Final feature matrix contains invalid values")

    baseline = [
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
        "month_sin", "month_cos", "horizon_hour", "price",
        "price_week_mean", "price_week_std", "price_week_percentile",
        "price_minus_week_mean", "water_value_end_mean",
        "price_minus_water_value_mean", "system_filling_ratio",
        "inflow_total", "inflow_total_24h_mean", "min_flow_total",
        "min_volume_total", "variable_constraint_active",
    ]

    def grouped(name):
        return [
            feature for feature, group in feature_group.items()
            if group == name and feature in X_all.columns
        ]

    def unique_existing(names):
        return list(dict.fromkeys(name for name in names if name in X_all.columns))

    feature_sets = {
        "baseline": unique_existing(baseline),
        "baseline_plus_price": unique_existing(baseline + grouped("price_windows")),
        "baseline_plus_price_water": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
        ),
        "baseline_plus_price_water_hydrology": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology")
        ),
        "full_engineered": unique_existing(
            baseline + grouped("price_windows") + grouped("water_margins")
            + grouped("hydrology") + grouped("constraints")
            + grouped("interactions")
        ),
    }

    generator_rows = []
    generator_config = topology["model"]["generator"]
    for unit in units:
        config = generator_config[unit]
        plant = unit.rsplit("_G", 1)[0]
        number_match = re.search(r"_G(\d+)$", unit)
        generator_rows.append({
            "unit": unit,
            "plant": plant,
            "generator_number": int(number_match.group(1)) if number_match else 1,
            "p_min": config.get("p_min", np.nan),
            "p_max": config.get("p_max", np.nan),
            "p_nom": config.get("p_nom", np.nan),
            "minimum_output_fraction": config.get("p_min", np.nan) / config.get("p_max", np.nan),
            "start_cost": config.get("startcost_const", 0.0),
            "stop_cost": config.get("stopcost_const", 0.0),
            "penstock": str(config.get("penstock", "unknown")),
        })
    generator_features = pd.DataFrame(generator_rows)
    generator_features["units_in_plant"] = (
        generator_features.groupby("plant")["unit"].transform("count")
    )

    return ProjectData(
        root=root,
        X_all=X_all,
        target_cube=target_cube,
        units=units,
        train_cases=train_cases,
        test_cases=test_cases,
        sample_submission=sample_submission,
        feature_sets=feature_sets,
        feature_group=feature_group,
        generator_features=generator_features,
        folds=make_purged_folds(train_cases),
    )

## physics — definitions only

In [4]:
"""Deterministic input-only features for the next hydropower experiments.

No fitted transforms, labels, predicted/observed future storage, or solver outputs.
Water budgets deliberately exclude unknown dispatch and are explicitly proxies.
"""

import numpy as np
import pandas as pd
import yaml



def window_sum(values, width, forward=True):
    """Truncated within-case sum, including the current hour; no edge repeats."""
    if not forward:
        return window_sum(values[:, ::-1], width, True)[:, ::-1]
    cumulative = np.pad(np.cumsum(values, axis=1, dtype=np.float64), ((0, 0), (1, 0)))
    left = np.arange(values.shape[1])
    right = np.minimum(left + width, values.shape[1])
    return cumulative[:, right] - cumulative[:, left]


def consecutive_true(mask):
    """Run length ending/starting at each position, reset at each case boundary."""
    past = np.zeros(mask.shape, dtype=np.float32)
    future = np.zeros_like(past)
    for h in range(mask.shape[1]):
        past[:, h] = mask[:, h] * (1 + (past[:, h - 1] if h else 0))
    for h in range(mask.shape[1] - 1, -1, -1):
        future[:, h] = mask[:, h] * (1 + (future[:, h + 1] if h + 1 < mask.shape[1] else 0))
    return past, future


def plant_sources(topology, plant):
    """Follow reservoir -> plant or reservoir -> tunnel -> plant connections.

    Stop at the first reservoir; do not pretend all upstream water arrives instantly.
    """
    connections = topology['connections']
    found = set()

    def visit(name, kind, seen):
        if (name, kind) in seen:
            return
        seen = seen | {(name, kind)}
        for edge in connections:
            if edge['to'] != name or edge['to_type'] != kind:
                continue
            if edge['from_type'] == 'reservoir':
                found.add(edge['from'])
            elif edge['from_type'] == 'tunnel':
                visit(edge['from'], 'tunnel', seen)

    visit(plant, 'plant', set())
    if not found:
        raise ValueError(f'No source reservoir found for {plant}')
    return sorted(found)


def build_research_features(data):
    """Return global additions, per-plant additions, and an auditable source mapping."""
    root = data.root
    with open(root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    starts = pd.DatetimeIndex(pd.concat([data.train_cases, data.test_cases])['starttime'])
    n = data.n_all_cases
    horizon = np.arange(HORIZON)
    time_index = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(horizon, n), unit='h')
    price = data.X_all['price'].to_numpy().reshape(n, HORIZON)
    volume = read_time_csv(root / 'data/kernel/Historical_volume_2015_2024.csv').set_index('date').reindex(starts)
    water = read_time_csv(root / 'data/kernel/Synthetic_water_value_2015_2024.csv').set_index('date').reindex(starts + pd.Timedelta(days=7))
    inflow = read_time_csv(root / 'data/kernel/Historical_inflow_1958_2025.csv').set_index('date').reindex(time_index)
    minimum = read_time_csv(root / 'data/extended/Constraint_min_volume.csv').set_index('date').reindex(time_index)
    extra = {}

    def add(target, name, values):
        values = np.broadcast_to(values, (n, HORIZON)).reshape(-1).astype(np.float32)
        if not np.isfinite(values).all():
            raise ValueError(f'Non-finite research feature {name}')
        target[name] = values

    add(extra, 'v2_day_in_horizon', np.broadcast_to(horizon // 24, (n, HORIZON)))
    add(extra, 'v2_hours_remaining', HORIZON - horizon)
    day_price = price.reshape(n, 7, 24)
    day_mean = np.repeat(day_price.mean(axis=2), 24, axis=1)
    day_rank = pd.DataFrame(day_price.reshape(-1, 24)).rank(axis=1, pct=True).to_numpy().reshape(n, HORIZON)
    add(extra, 'v2_price_day_rank', day_rank)
    add(extra, 'v2_price_minus_day_mean', price - day_mean)
    prefix_mean = np.cumsum(price, axis=1) / (horizon + 1)
    suffix_mean = np.cumsum(price[:, ::-1], axis=1)[:, ::-1] / (HORIZON - horizon)
    add(extra, 'v2_price_minus_prefix_mean', price - prefix_mean)
    add(extra, 'v2_price_minus_suffix_mean', price - suffix_mean)
    add(extra, 'v2_best_remaining_price_gap', np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1] - price)
    reservoirs = topology['model']['reservoir']
    reservoir_inputs = {}
    for name, config in reservoirs.items():
        cap = float(config['max_vol'])
        initial = volume[name].to_numpy()[:, None]
        terminal = water[name].to_numpy()[:, None]
        flow = inflow[name].to_numpy().reshape(n, HORIZON)
        # m3/s * 3600 / 1e6 -> million m3 per hour, matching storage units.
        hourly_volume = flow * 0.0036
        cumulative = np.cumsum(hourly_volume, axis=1)
        remaining = np.cumsum(hourly_volume[:, ::-1], axis=1)[:, ::-1]
        floor = minimum[name].to_numpy().reshape(n, HORIZON) if name in minimum else np.zeros_like(price)
        budget = initial + cumulative
        add(extra, f'v2_cumulative_inflow_fill__{name}', cumulative / cap)
        add(extra, f'v2_remaining_inflow_fill__{name}', remaining / cap)
        add(extra, f'v2_no_release_excess__{name}', np.maximum(budget - cap, 0) / cap)
        add(extra, f'v2_no_release_min_slack__{name}', (budget - floor) / cap)
        curve = config['vol_head']
        head = np.interp(initial[:, 0], curve['x'], curve['y'])[:, None]
        reservoir_inputs[name] = (cap, initial, terminal, cumulative, remaining, head)

    plants = {}
    mappings = []
    for plant, plant_config in topology['model']['plant'].items():
        sources = plant_sources(topology, plant)
        values = [reservoir_inputs[s] for s in sources]
        capacity = sum(v[0] for v in values)
        initial = sum(v[1] for v in values)
        # Multi-reservoir summaries are capacity-weighted approximations, not SHOP prices.
        terminal = sum(v[2] * v[0] for v in values) / capacity
        head = sum(v[5] * v[0] for v in values) / capacity - float(plant_config['outlet_line'])
        cum = sum(v[3] for v in values)
        remain = sum(v[4] for v in values)
        margin = price - terminal
        out = {}
        add(out, 'local_initial_fill', initial / capacity)
        add(out, 'local_gross_head_proxy', head)
        add(out, 'local_terminal_water_value', terminal)
        add(out, 'local_margin', margin)
        add(out, 'local_margin_normalized', margin / (np.abs(terminal) + price.std(axis=1)[:, None] + 1))
        add(out, 'local_cumulative_inflow_fill', cum / capacity)
        add(out, 'local_remaining_inflow_fill', remain / capacity)
        add(out, 'local_margin_x_fill', margin * initial / capacity)
        positive = np.maximum(margin, 0)
        negative = np.maximum(-margin, 0)
        past, future = consecutive_true(margin > 0)
        add(out, 'local_profitable_run_past', past)
        add(out, 'local_profitable_run_future', future)
        add(out, 'local_profitable_fraction_week', (margin > 0).mean(axis=1)[:, None])
        for hours in (6, 24):
            add(out, f'local_positive_margin_next_{hours}h', window_sum(positive, hours))
            add(out, f'local_negative_margin_next_{hours}h', window_sum(negative, hours))
            add(out, f'local_positive_margin_past_{hours}h', window_sum(positive, hours, False))
        # Start cost / rated production gives EUR/MWh; margin sums are hourly
        # EUR/MWh contributions. This is an approximate rated-power payback ratio.
        unit_configs = [topology['model']['generator'][u] for u in data.units if u.rsplit('_G', 1)[0] == plant]
        cost_per_mw = np.mean([(v.get('startcost_const', 0) + v.get('stopcost_const', 0)) / v['p_max'] for v in unit_configs])
        add(out, 'local_start_stop_payback_proxy', window_sum(positive, 24) / max(cost_per_mw, 1e-6))
        plants[plant] = pd.DataFrame(out)
        mappings.append({'plant': plant, 'source_reservoirs': '|'.join(sources), 'source_capacity_Mm3': capacity})
    return pd.DataFrame(extra), plants, pd.DataFrame(mappings)

## fit_helpers — definitions only

In [5]:
"""Sequential, resumable challenger. Importing this module never trains models."""
import gc
import hashlib
import importlib.metadata
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits


def raw_digest(root):
    paths = sorted((root / 'data').rglob('*.csv')) + sorted((root / 'data').rglob('*.yaml'))
    paths += [root / 'prediction_mapping.csv', root / 'sample_submission.csv']
    digest = hashlib.sha256()
    for path in paths:
        digest.update(str(path.relative_to(root)).encode())
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
    return digest.hexdigest()


def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix('.tmp.json')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    os.replace(temporary, path)


def purged_year(data, year):
    starts = pd.DatetimeIndex(data.train_cases['starttime'])
    begin = pd.Timestamp(f'{year}-01-01', tz='UTC')
    end = pd.Timestamp(f'{year+1}-01-01', tz='UTC')
    train = np.flatnonzero(starts + pd.Timedelta(hours=HORIZON-1) < begin)
    valid = np.flatnonzero((starts >= begin) & (starts < end))
    if not len(train) or not len(valid):
        raise ValueError('Empty temporal split')
    assert starts[train].max() + pd.Timedelta(hours=HORIZON-1) < starts[valid].min()
    return train, valid


def score_cube(truth, prediction):
    return float(roc_auc_score(truth.ravel(), prediction.ravel()))


def fit_member(spec, X, y, rounds, threads, valid=None):
    parameters = dict(spec['parameters'])
    if spec['backend'] == 'xgboost':
        from xgboost import XGBClassifier
        model = XGBClassifier(**parameters, n_estimators=int(rounds), n_jobs=threads,
                              **({'early_stopping_rounds': 80} if valid is not None else {}))
        model.fit(X, y, eval_set=[valid] if valid is not None else None, verbose=False)
        best = int(model.best_iteration) + 1 if valid is not None else int(rounds)
    else:
        import lightgbm as lgb
        parameters['num_threads'] = threads
        training = lgb.Dataset(X, label=y, free_raw_data=True)
        options = {}
        if valid is not None:
            parameters['metric'] = 'auc'
            options = dict(valid_sets=[lgb.Dataset(valid[0], label=valid[1], reference=training)],
                           callbacks=[lgb.early_stopping(80, verbose=False)])
        model = lgb.train(parameters, training, num_boost_round=int(rounds), **options)
        best = int(model.best_iteration or rounds)
    return model, best

## competition — definitions only

In [6]:
"""Feature-only refinements for the frozen notebook 12/13 estimators."""
import numpy as np
import pandas as pd
import yaml


def price_competition(price, widths=(24, 72, 168)):
    """Strictly better alternative hours within this case, excluding current hour.

    Compute relative position and mean forgone price premium separately before
    and after the current hour. No edge replication or cross-case windows.
    """
    price = np.asarray(price, dtype=np.float64)
    n, horizon = price.shape
    ahead_count = np.zeros_like(price); past_count = np.zeros_like(price)
    ahead_gain = np.zeros_like(price); past_gain = np.zeros_like(price)
    scale = price.std(axis=1)[:, None] + 1.
    output = {}
    for distance in range(1, max(widths)):
        if distance < horizon:
            gap = price[:, distance:] - price[:, :-distance]
            ahead_count[:, :-distance] += gap > 0
            past_count[:, distance:] += gap < 0
            ahead_gain[:, :-distance] += np.maximum(gap, 0)
            past_gain[:, distance:] += np.maximum(-gap, 0)
        width = distance + 1
        if width in widths:
            ahead_n = np.maximum(1, np.minimum(width-1, horizon-1-np.arange(horizon)))
            past_n = np.maximum(1, np.minimum(width-1, np.arange(horizon)))
            output[f'v4_better_future_fraction_{width}'] = ahead_count / ahead_n
            output[f'v4_better_past_fraction_{width}'] = past_count / past_n
            output[f'v4_future_premium_{width}'] = ahead_gain / ahead_n / scale
            output[f'v4_past_premium_{width}'] = past_gain / past_n / scale
    return {k: v.astype(np.float32) for k, v in output.items()}


def build_refinement_features(data):
    n = data.n_all_cases
    price = data.X_all['price'].to_numpy().reshape(n, HORIZON)
    common = pd.DataFrame({k: v.ravel() for k, v in price_competition(price).items()})
    starts = pd.DatetimeIndex(pd.concat([data.train_cases, data.test_cases])['starttime'])
    times = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(np.arange(HORIZON), n), unit='h')
    with open(data.root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    volumes = read_time_csv(data.root / 'data/kernel/Historical_volume_2015_2024.csv').set_index('date').reindex(starts)
    water = read_time_csv(data.root / 'data/kernel/Synthetic_water_value_2015_2024.csv').set_index('date').reindex(starts + pd.Timedelta(days=7))
    inflow = read_time_csv(data.root / 'data/kernel/Historical_inflow_1958_2025.csv').set_index('date').reindex(times)
    minimum = read_time_csv(data.root / 'data/extended/Constraint_min_volume.csv').set_index('date').reindex(times)
    price_only, combined, audit = {}, {}, []
    for plant in topology['model']['plant']:
        sources = plant_sources(topology, plant)
        cap = sum(topology['model']['reservoir'][s]['max_vol'] for s in sources)
        terminal = sum(water[s].to_numpy()[:, None] * topology['model']['reservoir'][s]['max_vol'] for s in sources) / cap
        margin = price - terminal
        flow = inflow[sources].sum(axis=1).to_numpy().reshape(n, HORIZON)
        initial = volumes[sources].sum(axis=1).to_numpy()[:, None]
        floor = sum((minimum[s].to_numpy().reshape(n, HORIZON) for s in sources if s in minimum), np.zeros_like(price))
        out = {}

        def add(name, values):
            values = np.broadcast_to(values, (n, HORIZON)).ravel().astype(np.float32)
            if not np.isfinite(values).all():
                raise ValueError(f'Non-finite feature {plant}/{name}')
            out['v4_' + name] = values

        prefix = np.arange(1, HORIZON+1)
        suffix = prefix[::-1]
        positive = margin > 0
        add('profitable_prefix_fraction', np.cumsum(positive, axis=1) / prefix)
        add('profitable_suffix_fraction', np.cumsum(positive[:, ::-1], axis=1)[:, ::-1] / suffix)
        inflow_cumulative = np.cumsum(flow, axis=1)
        flow_total = flow.sum(axis=1)[:, None]
        fraction_arrived = np.divide(inflow_cumulative, flow_total,
                                     out=np.zeros_like(inflow_cumulative), where=flow_total > 0)
        profit = np.maximum(margin, 0)
        profit_total = profit.sum(axis=1)[:, None]
        fraction_profit_elapsed = np.divide(np.cumsum(profit, axis=1), profit_total,
                                            out=np.zeros_like(profit), where=profit_total > 0)
        add('inflow_arrival_fraction', fraction_arrived)
        add('profit_elapsed_fraction', fraction_profit_elapsed)
        add('inflow_minus_profit_timing', fraction_arrived - fraction_profit_elapsed)
        add('no_profitable_hours', profit_total == 0)
        add('no_inflow', flow_total == 0)
        for width in (24, 72, 168):
            flow_window = window_sum(flow, width)
            weighted = np.divide(window_sum(flow*price, width), flow_window,
                                 out=np.zeros_like(price, dtype=np.float64), where=flow_window > 0)
            add(f'inflow_weighted_price_gap_{width}', np.where(flow_window > 0, weighted-price, 0))
            add(f'profitable_inflow_fill_{width}', window_sum(flow*positive, width)*.0036/cap)
        future_floor = np.maximum.accumulate(floor[:, ::-1], axis=1)[:, ::-1]
        add('upcoming_floor_increase_fill', (future_floor-floor)/cap)
        add('initial_slack_to_future_floor', (initial-future_floor)/cap)
        add('hours_until_highest_remaining_price', _hours_to_future_max(price))
        add('profitable_hours_remaining', np.cumsum(positive[:, ::-1], axis=1)[:, ::-1])
        water_features = pd.DataFrame(out)
        price_only[plant] = common
        combined[plant] = pd.concat([common, water_features], axis=1)
        audit.append(dict(plant=plant, sources='|'.join(sources),
                          price_features=len(common.columns), timing_features=len(water_features.columns)))
    return {'competition': price_only, 'competition_timing': combined}, pd.DataFrame(audit)


def _hours_to_future_max(price):
    n, horizon = price.shape
    result = np.zeros_like(price)
    best = np.full(n, -np.inf)
    index = np.full(n, horizon-1)
    for t in range(horizon-1, -1, -1):
        update = price[:, t] >= best
        best = np.maximum(best, price[:, t])
        index = np.where(update, t, index)
        result[:, t] = index-t
    return result

## paths — definitions only

In [7]:
"""Fixed, input-only whole-week descriptors for notebook 18.

No labels, fitted transforms, future observed storage, or cross-case windows.
The supplied future price/inflow paths are inputs to each scheduling problem.
"""
import numpy as np
import pandas as pd
import yaml
from scipy.fft import dct


def contiguous_competition(price):
    """Consecutive strictly better hours immediately before/after each hour.

    Unlike a rank, an interruption ends the sequence. Current hour is excluded.
    Equal prices end a run. Each case is independent.
    """
    price = np.asarray(price)
    n, h = price.shape
    result = {}
    for label, values in [('next', price), ('past', price[:, ::-1])]:
        active = np.ones((n, h), dtype=bool)
        count = np.zeros((n, h), np.float32)
        premium = np.zeros((n, h), np.float32)
        for shift in range(1, h):
            gap = values[:, shift:] - values[:, :-shift]
            active[:, :-shift] &= gap > 0
            count[:, :-shift] += active[:, :-shift]
            premium[:, :-shift] += np.maximum(gap, 0) * active[:, :-shift]
        result[f'path_consecutive_better_{label}'] = count if label=='next' else count[:, ::-1]
        result[f'path_contiguous_premium_{label}'] = premium if label=='next' else premium[:, ::-1]
    return result


def weekly_shape(price):
    """Daily and low-frequency shape, preserving absolute day positions in a case."""
    price = np.asarray(price, dtype=np.float64)
    n, h = price.shape
    if h != HORIZON:
        raise ValueError('Expected exactly 168 hourly prices')
    scale = price.std(axis=1)[:, None] + 1.
    centered = (price-price.mean(axis=1)[:, None])/scale
    out = {}
    daily = price.reshape(n, 7, 24)
    for day in range(7):
        # Compare current hour with EACH day, not just its own day or whole week.
        out[f'path_day{day}_mean_gap'] = (daily[:, day].mean(axis=1)[:, None]-price)/scale
        out[f'path_day{day}_peak_gap'] = (daily[:, day].max(axis=1)[:, None]-price)/scale
    coefficients = dct(centered, type=2, norm='ortho', axis=1)
    for k in range(1, 7):
        out[f'path_price_dct_{k}'] = np.broadcast_to(coefficients[:, k, None], (n,h))
    for forward, label in [(True,'next'), (False,'past')]:
        for width in (48, 96, 168):
            counts = np.minimum(width, np.arange(h,0,-1) if forward else np.arange(1,h+1))
            average = window_sum(price,width,forward)/counts
            out[f'path_price_{label}_{width}_relative'] = (price-average)/scale
    for name, values in contiguous_competition(price).items():
        out[name] = values/scale if 'premium' in name else values
    return {name: np.asarray(value, dtype=np.float32) for name,value in out.items()}


def build_path_features(data, local):
    n = data.n_all_cases
    price = data.X_all['price'].to_numpy().reshape(n,HORIZON)
    global_frame = pd.DataFrame({k:v.ravel() for k,v in weekly_shape(price).items()})
    with open(data.root/'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    frames, records = {}, []
    for plant in local:
        sources = plant_sources(topology,plant)
        cap = sum(topology['model']['reservoir'][r]['max_vol'] for r in sources)
        terminal = local[plant]['local_terminal_water_value'].to_numpy().reshape(n,HORIZON)[:,0,None]
        margin = price-terminal
        scale = price.std(axis=1)[:,None]+np.abs(terminal)+1.
        daily_margin = margin.reshape(n,7,24)
        # All inputs come from X_all, already restricted to the supplied case window.
        flow = sum(data.X_all[f'inflow__{r}'].to_numpy().reshape(n,HORIZON) for r in sources)
        daily_flow = flow.reshape(n,7,24)
        extra = {}
        def add(name, value):
            extra['path_'+name] = np.broadcast_to(value,(n,HORIZON)).ravel().astype(np.float32)
        for day in range(7):
            add(f'day{day}_positive_margin', np.maximum(daily_margin[:,day],0).sum(axis=1)[:,None]/scale)
            add(f'day{day}_negative_margin', np.maximum(-daily_margin[:,day],0).sum(axis=1)[:,None]/scale)
            add(f'day{day}_source_inflow_fill',daily_flow[:,day].sum(axis=1)[:,None]*.0036/cap)
        profit=np.maximum(margin,0)
        profit_sum=profit.sum(axis=1)[:,None]
        shares=np.divide(profit,profit_sum,out=np.zeros_like(profit),where=profit_sum>0)
        add('profit_concentration', (shares**2).sum(axis=1)[:,None])
        add('profit_center_of_time', (shares*np.arange(HORIZON)).sum(axis=1)[:,None]/(HORIZON-1))
        add('current_profit_share',shares)
        frame=pd.concat([global_frame,pd.DataFrame(extra)],axis=1)
        if not np.isfinite(frame.to_numpy()).all() or not frame.columns.is_unique:
            raise ValueError(f'Invalid path features for {plant}')
        frames[plant]=frame
        records.append(dict(plant=plant,sources='|'.join(sources),global_features=len(global_frame.columns),local_features=len(extra)))
    return frames,pd.DataFrame(records)

## staged — definitions only

In [8]:
"""Bounded staged boosting experiment. Importing this module never trains."""
import gc
import hashlib
import importlib.metadata
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits


def factor_key(factor):
    return f'{float(factor):g}'


def scaled_rounds(original, factor):
    if not 0 < float(factor) <= 2:
        raise ValueError('Round multiplier must be in (0, 2]')
    return max(1, int(round(int(original)*float(factor))))


def checked_cube(values, cases, units):
    values=np.asarray(values,dtype=np.float32)
    if values.shape != (cases,units,HORIZON) or not np.isfinite(values).all():
        raise ValueError('Invalid prediction dimensions or non-finite values')
    if values.min()<0 or values.max()>1:
        raise ValueError('Prediction outside [0,1]')
    return values


def staged_predict(model, backend, X, rounds, threads):
    if backend=='xgboost':
        return model.predict_proba(X,iteration_range=(0,int(rounds)))[:,1]
    return model.predict(X,num_iteration=int(rounds),num_threads=threads)

## hydro_lp — definitions only

In [9]:
"""Weekly linear relaxation of the Tokke-Vinje watercourse; input-only features, never fits a model.

One LP per scheduling case: maximise price revenue plus SHOP-style end value of stored water,
subject to reservoir balances, plant/tunnel/river limits, minimum flows and soft minimum volumes.
Uses only the case's supplied price/inflow path, initial volume, terminal water value, constraint
series and topology. No labels, no SHOP results, no observed storage after case start.
Deliberate simplifications (the boosted model corrects them): linear best-point conversion at the
initial head, no p_min/start costs, no head dependence, tunnels as free transfers up to a cap.
"""
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml
from scipy.optimize import linprog

MM3 = 0.0036  # Mm3 per (m3/s during one hour)
# SHOP: energy_value_input is local, relative to the reservoir below the FIRST plant under the
# reservoir. This map reproduces the identical water-value groups found in the raw data.
FIRST_PLANT = {'Bandak': 'Hogga', 'Vinjevatn': 'Tokke', 'Byrtevatn': 'Lio', 'Botnedalsvatn': 'Byrte',
               'Vaamarvatn': 'Vinje', 'Totak': 'Vinje', 'Venemo': 'Vinje', 'Hyljelihyl': 'Vinje',
               'Vatjern': 'Haukeli', 'Langeidvatn': 'Haukeli', 'Songavatn': 'Songa', 'Bitdalsvatn': 'Songa',
               'Foersvatn': 'Kjela', 'Bordalsvatn': 'Kjela', 'Langesae': 'Kjela',
               'Kjelavatn': 'Vesle Kjela', 'Staavatn': 'Vesle Kjela'}
PLANT_SOURCE = {'Hogga': 'Bandak', 'Tokke': 'Vinjevatn', 'Lio': 'Byrtevatn', 'Byrte': 'Botnedalsvatn',
                'Vinje': 'Vaamarvatn', 'Haukeli': 'Vatjern', 'Songa': 'Songavatn', 'Kjela': 'Foersvatn',
                'Vesle Kjela': 'Kjelavatn'}
MIN_FLOW_PENALTY = 5000.   # EUR per (m3/s h) of unmet minimum flow; keeps every case feasible
MIN_VOLUME_PENALTY = 1e4   # EUR per (Mm3 h) below a minimum volume (SHOP: soft constraint)
TUNNEL_CAP, TUNNEL_COST = 300., .01  # ponytail: free transfer, no level-dependent tunnel hydraulics


class HydroLP:
    """Static sparse structure is built once; each case changes costs, right-hand sides and bounds."""

    def __init__(self, topology, min_flow_rivers, min_volume_reservoirs):
        model, edges = topology['model'], topology['connections']
        self.model = model
        self.res = list(model['reservoir'])
        self.plants = list(model['plant'])
        self.ridx = {r: i for i, r in enumerate(self.res)}
        down = {}
        for e in edges:
            down.setdefault((e['from'], e['from_type']), []).append((e['to'], e['to_type']))

        def resolve(node, kind, via=False):
            # Follow rivers until a reservoir; r_Vest_Vassdraget is a junction ending in Bandak.
            if kind == 'reservoir':
                return node, via
            following = down.get((node, kind), [])
            if not following:
                return 'SEA', via
            (nxt, nxt_kind), = following
            return resolve(nxt, nxt_kind, via or node == 'r_Vest_Vassdraget')

        arcs = []
        generators = model['generator']
        for p in self.plants:
            units = [g for g in generators if g.rsplit('_G', 1)[0] == p]
            qmax = float(sum(max(max(c['x']) for c in generators[g]['turb_eff_curves']) for g in units))
            outs = down.get((p, 'plant'), [])
            dst, via = resolve(*outs[0]) if outs else ('SEA', False)
            sources = ('Songavatn', 'Bitdalsvatn') if p == 'Songa' else (PLANT_SOURCE[p],)
            for s in sources:  # Songa draws from two tunnel-connected reservoirs
                arcs.append(dict(name=f'plant:{p}:{s}', src=s, dst=dst, cost=0., ub=qmax, plant=p, via=via,
                                 units=len(units)))
        for name, cfg in model['river'].items():
            ups = [e['from'] for e in edges if e['to'] == name and e['to_type'] == 'river' and e['from_type'] == 'reservoir']
            if not ups:
                continue
            (src,) = ups
            dst, via = resolve(name, 'river')
            if 'flow_cost_curve' in cfg:  # piecewise-constant marginal cost, NOK|EUR per (m3/s h)
                x, y = cfg['flow_cost_curve']['x'], cfg['flow_cost_curve']['y']
                for k in range(len(x) - 1):
                    arcs.append(dict(name=f'river:{name}:{k}', src=src, dst=dst, cost=float(y[k]),
                                     ub=float(x[k + 1] - x[k]), plant=None, via=via))
            else:
                arcs.append(dict(name=f'river:{name}', src=src, dst=dst, cost=float(cfg.get('flow_cost_const', 0.)),
                                 ub=float(cfg.get('max_flow_const', np.inf)), plant=None, via=via))
        for name in model['tunnel']:
            a, b = name.split('_', 1)
            if b in model['plant']:
                continue
            for s, d in ((a, b), (b, a)):
                arcs.append(dict(name=f'tunnel:{s}>{d}', src=s, dst=d, cost=TUNNEL_COST, ub=TUNNEL_CAP,
                                 plant=None, via=False))
        self.arcs = arcs
        self.below = {a['plant']: a['dst'] for a in arcs if a['plant']}
        self.plant_arc = {p: [k for k, a in enumerate(arcs) if a['plant'] == p] for p in self.plants}
        self.min_flow_rivers = list(min_flow_rivers)
        self.min_volume_reservoirs = [r for r in min_volume_reservoirs if r in self.ridx]
        T, R, A = HORIZON, len(self.res), len(arcs)
        F, M = len(self.min_flow_rivers), len(self.min_volume_reservoirs)
        self.nf, self.nv = A * T, R * T
        self.nvar = self.nf + self.nv + F * T + M * T
        t = np.arange(T)
        # Balance: V[t+1] - V[t] + MM3*(outflow - inflow) = MM3*natural_inflow (+ V0 at t=0).
        rows = [np.repeat(np.arange(R) * T, T) + np.tile(t, R)]
        cols = [self.nf + rows[0]]
        vals = [np.ones(R * T)]
        prev = np.repeat(np.arange(R) * T, T - 1) + np.tile(t[1:], R)
        rows.append(prev); cols.append(self.nf + prev - 1); vals.append(-np.ones(len(prev)))
        for k, arc in enumerate(arcs):
            for node, sign in ((arc['src'], 1.), (arc['dst'], -1.)):
                if node in self.ridx:
                    rows.append(self.ridx[node] * T + t); cols.append(k * T + t); vals.append(np.full(T, sign * MM3))
        self.A_eq = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                                  shape=(R * T, self.nvar))
        rows, cols, vals, row = [], [], [], 0
        for k, river in enumerate(self.min_flow_rivers):
            members = [a for a, arc in enumerate(arcs)
                       if (arc['via'] if river == 'r_Vest_Vassdraget' else arc['name'] == f'river:{river}')]
            if not members:
                raise ValueError(f'No arcs for minimum-flow river {river}')
            for a in members:
                rows.append(row + t); cols.append(a * T + t); vals.append(-np.ones(T))
            rows.append(row + t); cols.append(self.nf + self.nv + k * T + t); vals.append(-np.ones(T))
            row += T
        for k, name in enumerate(self.min_volume_reservoirs):
            rows.append(row + t); cols.append(self.nf + self.ridx[name] * T + t); vals.append(-np.ones(T))
            rows.append(row + t); cols.append(self.nf + self.nv + F * T + k * T + t); vals.append(-np.ones(T))
            row += T
        self.capacity_rows = []
        for p, members in self.plant_arc.items():
            if len(members) > 1:  # shared turbine capacity for multi-intake plants
                for a in members:
                    rows.append(row + t); cols.append(a * T + t); vals.append(np.ones(T))
                self.capacity_rows.append((row, arcs[members[0]]['ub']))
                row += T
        self.A_ub = sp.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
                                  shape=(row, self.nvar))
        self.lb = np.zeros(self.nvar)
        self.ub_template = np.full(self.nvar, np.inf)
        for k, arc in enumerate(arcs):
            self.ub_template[k * T:(k + 1) * T] = arc['ub']
        self.cost_template = np.zeros(self.nvar)
        for k, arc in enumerate(arcs):
            self.cost_template[k * T:(k + 1) * T] = arc['cost']
        self.cost_template[self.nf + self.nv:self.nf + self.nv + F * T] = MIN_FLOW_PENALTY
        self.cost_template[self.nf + self.nv + F * T:] = MIN_VOLUME_PENALTY

    def conversion(self, plant, level):
        """MW per m3/s at one unit's best point and the given gross level (SHOP converts WV here)."""
        cfg = self.model['plant'][plant]
        gens = [g for g in self.model['generator'] if g.rsplit('_G', 1)[0] == plant]
        head = level - float(cfg['outlet_line'])
        curve = min(self.model['generator'][gens[0]]['turb_eff_curves'], key=lambda c: abs(c['ref'] - head))
        k = int(np.argmax(curve['y']))
        q, eta = float(curve['x'][k]), float(curve['y'][k]) / 100
        penstock = max([float(v) for v in cfg.get('penstock_loss') or [0.]])
        loss = float(cfg['main_loss'][0]) * (q * len(gens)) ** 2 + penstock * q ** 2
        return 9.81e-3 * max(head - loss, 1.) * eta * .98

    def solve(self, price, inflow, v0, wv_end, min_flow, min_volume):
        """price (T,); inflow: node -> (T,); v0/wv_end: reservoir -> float;
        min_flow: river -> (T,); min_volume: reservoir -> (T,). Returns plant x 6 x T array."""
        T, R = HORIZON, len(self.res)
        level = {r: np.interp(v0[r], self.model['reservoir'][r]['vol_head']['x'],
                              self.model['reservoir'][r]['vol_head']['y']) for r in self.res}
        e = {p: self.conversion(p, level[PLANT_SOURCE[p]]) for p in self.plants}
        G = {}

        def global_value(r):  # EUR per Mm3: local value x conversion + value below the plant
            if r == 'SEA':
                return 0.
            if r not in G:
                p = FIRST_PLANT[r]
                G[r] = wv_end[r] * e[p] / MM3 + global_value(self.below[p])
            return G[r]
        c = self.cost_template.copy()
        for k, arc in enumerate(self.arcs):
            if arc['plant']:
                c[k * T:(k + 1) * T] -= price * e[arc['plant']]
        ub = self.ub_template.copy()
        for i, r in enumerate(self.res):
            c[self.nf + i * T + T - 1] -= global_value(r)
            ub[self.nf + i * T:self.nf + (i + 1) * T] = max(float(self.model['reservoir'][r]['max_vol']), v0[r])
        b_eq = np.zeros(R * T)
        for i, r in enumerate(self.res):
            b_eq[i * T:(i + 1) * T] = MM3 * inflow[r]
            b_eq[i * T] += v0[r]
        b_eq[self.ridx['Bandak'] * T:(self.ridx['Bandak'] + 1) * T] += MM3 * inflow['r_Vest_Vassdraget']
        b_ub = []
        for river in self.min_flow_rivers:
            natural = inflow['r_Vest_Vassdraget'] if river == 'r_Vest_Vassdraget' else 0.
            b_ub.append(-(min_flow[river] - natural))
        for r in self.min_volume_reservoirs:
            b_ub.append(-min_volume[r])
        for _, cap in self.capacity_rows:
            b_ub.append(np.full(T, cap))
        out = linprog(c, A_ub=self.A_ub, b_ub=np.concatenate(b_ub), A_eq=self.A_eq, b_eq=b_eq,
                      bounds=np.column_stack([self.lb, ub]), method='highs-ds')
        if out.status != 0:
            raise RuntimeError(f'LP failed: {out.message}')
        x = out.x
        lam = -out.eqlin.marginals.reshape(R, T)  # value of one extra Mm3 at (reservoir, hour)
        reduced = out.lower.marginals + out.upper.marginals  # objective change per unit flow
        result = np.empty((len(self.plants), 6, T))
        for j, p in enumerate(self.plants):
            members = self.plant_arc[p]
            q = sum(x[a * T:(a + 1) * T] for a in members) / self.arcs[members[0]]['ub']
            s, d = self.ridx[PLANT_SOURCE[p]], self.below[p]
            lam_d = lam[self.ridx[d]] if d in self.ridx else np.zeros(T)
            vol = lambda i: np.concatenate([[v0[self.res[i]]], x[self.nf + i * T:self.nf + (i + 1) * T - 1]])
            result[j] = [q, MM3 * (lam[s] - lam_d) / e[p], -reduced[members[0] * T:(members[0] + 1) * T] / e[p],
                         vol(s), vol(self.ridx[d]) if d in self.ridx else np.zeros(T), np.full(T, e[p])]
        return result


def lp_inputs(root, starts):
    """Raw per-case inputs, exactly the supplied case window; no later observed storage."""
    with open(root / 'data/extended/Tokke_Vinje_topology.yaml') as stream:
        topology = yaml.safe_load(stream)
    starts = pd.DatetimeIndex(starts)
    n = len(starts)
    times = starts.repeat(HORIZON) + pd.to_timedelta(np.tile(np.arange(HORIZON), n), unit='h')

    def hourly(path):
        frame = read_time_csv(path).set_index('date').select_dtypes(include=np.number).reindex(times)
        if frame.isna().any().any():
            raise ValueError(f'Missing hourly inputs in {path.name}')
        return {c: frame[c].to_numpy(np.float64).reshape(n, HORIZON) for c in frame}

    def daily(path, when):
        frame = read_time_csv(path).set_index('date').select_dtypes(include=np.number).reindex(when)
        if frame.isna().any().any():
            raise ValueError(f'Missing daily inputs in {path.name}')
        return frame
    price = read_time_csv(root / 'data/kernel/Historical_day_ahead_price_2015_2025.csv').set_index('date')
    price = price['Day-ahead price (EUR/MWh)'].reindex(times).to_numpy(np.float64).reshape(n, HORIZON)
    return dict(topology=topology, price=price,
                inflow=hourly(root / 'data/kernel/Historical_inflow_1958_2025.csv'),
                min_flow=hourly(root / 'data/extended/Constraint_min_flow.csv'),
                min_volume=hourly(root / 'data/extended/Constraint_min_volume.csv'),
                v0=daily(root / 'data/kernel/Historical_volume_2015_2024.csv', starts),
                wv_end=daily(root / 'data/kernel/Synthetic_water_value_2015_2024.csv', starts + pd.Timedelta(days=7)))


def solve_all_cases(root, starts, threads=1):
    """Deterministic dual simplex per case; threads only change speed, not results.
    Returns (cases x plants x 6 x T cube, plant names, seconds)."""
    from concurrent.futures import ThreadPoolExecutor
    inputs = lp_inputs(root, starts)
    lp = HydroLP(inputs['topology'], inputs['min_flow'], inputs['min_volume'])
    n = len(inputs['price'])

    def one(k):
        return lp.solve(inputs['price'][k], {c: v[k] for c, v in inputs['inflow'].items()},
                        inputs['v0'].iloc[k].to_dict(), inputs['wv_end'].iloc[k].to_dict(),
                        {c: v[k] for c, v in inputs['min_flow'].items()},
                        {c: v[k] for c, v in inputs['min_volume'].items()})
    started = time.perf_counter()
    with ThreadPoolExecutor(max(1, int(threads))) as pool:
        cube = np.stack(list(pool.map(one, range(n)))).astype(np.float32)
    if not np.isfinite(cube).all():
        raise ValueError('Non-finite LP output')
    return cube, lp.plants, time.perf_counter() - started


def lp_plant_features(cube, plants, price, wv_source, units_per_plant):
    """Per-plant model inputs from the LP solution. wv_source: plant -> (cases,) terminal local value."""
    n = len(price)
    scale_price = price.std(axis=1)[:, None]
    frames = {}
    for j, p in enumerate(plants):
        q, threshold, rc_margin, vol_src, vol_dst, _ = (cube[:, j, k].astype(np.float64) for k in range(6))
        wv = np.asarray(wv_source[p], np.float64)[:, None]
        scale = scale_price + np.abs(wv) + 1.
        out = {
            'lp_discharge_fraction': q,
            'lp_week_fraction': np.broadcast_to(q.mean(axis=1, keepdims=True), q.shape),
            'lp_day_fraction': np.repeat(q.reshape(n, 7, 24).mean(axis=2), 24, axis=1),
            'lp_next_24h_fraction': window_sum(q, 24) / np.minimum(24, np.arange(HORIZON, 0, -1)),
            'lp_past_24h_fraction': window_sum(q, 24, False) / np.minimum(24, np.arange(1, HORIZON + 1)),
            'lp_reduced_cost_margin': rc_margin,
            'lp_reduced_cost_margin_normalized': rc_margin / scale,
            'lp_dual_margin': price - threshold,
            'lp_dual_margin_normalized': (price - threshold) / scale,
            'lp_threshold_shift_normalized': (threshold - wv) / scale,
            'lp_source_volume_mm3': vol_src,
            'lp_downstream_volume_mm3': vol_dst,
        }
        if units_per_plant[p] > 1:
            out['lp_units_equivalent'] = q * units_per_plant[p]
        frame = pd.DataFrame({k: np.asarray(v, np.float32).ravel() for k, v in out.items()})
        if not np.isfinite(frame.to_numpy()).all():
            raise ValueError(f'Non-finite LP feature for {p}')
        frames[p] = frame
    return frames

## experiment — definitions only

In [10]:
"""Controlled experiment: exact v5 model versus v5 + LP-relaxation features. Importing never trains.

Only the feature set differs between control and candidate: same estimators, parameters, seeds,
2x inherited tree counts and 50/50 XGBoost/LightGBM blend as multiscale_boosting_v5.csv.
Selection uses the development years; the audit year is looked at once, afterwards.
"""
import gc
import hashlib
import importlib.metadata
import json
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits

VARIANTS = ('v5', 'v5_lp')
FAMILIES = ('xgb', 'lgb')
FACTORS = (1., 1.5, 2.)       # prefixes of one fit; 1x/1.5x are diagnostics only
DECISION_FACTOR = 2.          # v5 used 2x the inherited per-unit counts
XGB_WEIGHT = .5               # v5 blend weight; not re-tuned here
MAX_DEV_YEAR_LOSS = .0005     # pre-registered: no development year may lose more than this


def append_csv(path, rows):
    frame = pd.DataFrame(rows)
    frame.to_csv(path, mode='a', header=not Path(path).exists(), index=False)


class LPRunner:
    def __init__(self, data, extra, local, competition, paths, lp, specs, rounds, threads,
                 source_digest, directory, use_checkpoints=True):
        self.data, self.local, self.competition, self.paths, self.lp = data, local, competition, paths, lp
        self.specs, self.rounds = specs, rounds
        self.threads = max(1, min(int(threads), os.cpu_count() or 1))
        self.use_checkpoints = bool(use_checkpoints)
        self.source_digest = source_digest
        self.base_names = list(data.feature_sets['full_engineered']) + list(extra.columns)
        self.base = np.column_stack([data.X_all[data.feature_sets['full_engineered']].to_numpy(np.float32),
                                     extra.to_numpy(np.float32)])
        self.raw = raw_digest(data.root)
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)

    def frames(self, plant, variant):
        if variant not in VARIANTS:
            raise ValueError(f'Unknown variant {variant}')
        blocks = [self.local[plant], self.competition[plant], self.paths[plant]]
        return blocks + [self.lp[plant]] if variant == 'v5_lp' else blocks

    def feature_names(self, plant, variant):
        names = self.base_names + [c for f in self.frames(plant, variant) for c in f.columns]
        if len(set(names)) != len(names):
            raise ValueError('Duplicate model features')
        return names

    def identity(self):
        return dict(source=self.source_digest, raw=self.raw, specs=self.specs, rounds=self.rounds,
                    packages={p: importlib.metadata.version(p) for p in
                              ('numpy', 'pandas', 'scipy', 'scikit-learn', 'xgboost', 'lightgbm')})

    def run(self, family, variant, train, predict, stage, factors=FACTORS):
        """One fit per unit, several tree-count prefixes. Completed units resume from disk."""
        train, predict = np.asarray(train, dtype=int), np.asarray(predict, dtype=int)
        if not len(train) or not len(predict) or len(np.intersect1d(train, predict)):
            raise ValueError('Empty or overlapping training/prediction cases')
        if train.max() >= self.data.n_train_cases or predict.max() >= self.data.n_all_cases or min(train.min(), predict.min()) < 0:
            raise ValueError('Case index outside dataset')
        factors = tuple(sorted({float(f) for f in factors}))
        names_hash = hashlib.sha256(json.dumps({p: self.feature_names(p, variant) for p in self.local}).encode()).hexdigest()
        manifest = dict(self.identity(), family=family, variant=variant, stage=stage, threads=self.threads,
                        features=names_hash, train=list(map(int, train)), predict=list(map(int, predict)),
                        factors=list(factors))
        signature = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:24]
        folder = self.directory / f'{stage}_{family}_{variant}' / signature
        folder.mkdir(parents=True, exist_ok=True)
        save_json(folder / 'manifest.json', manifest)
        cubes = {factor_key(f): np.empty((len(predict), len(self.data.units), HORIZON), np.float32) for f in factors}
        train_rows, pred_rows = rows_for_cases(train), rows_for_cases(predict)
        for i, unit in enumerate(self.data.units):
            started = time.perf_counter()
            counts = [scaled_rounds(self.rounds[family][unit], f) for f in factors]
            checkpoint = folder / f'unit_{i:02d}.npz'
            plant = unit.rsplit('_G', 1)[0]
            if self.use_checkpoints and checkpoint.exists():
                with np.load(checkpoint, allow_pickle=False) as saved:
                    if saved['rounds'].tolist() != counts:
                        raise ValueError(f'Mismatched checkpoint: {checkpoint}')
                    predictions = saved['predictions'].copy()
                status = 'restored'
            else:
                names = self.feature_names(plant, variant)
                X = np.ascontiguousarray(np.column_stack(
                    [self.base] + [f.to_numpy(np.float32) for f in self.frames(plant, variant)]), dtype=np.float32)
                y = self.data.target_cube[train, i].ravel()
                print(f'{stage} | {family}/{variant} | {unit} | {max(counts)} trees, {X.shape[1]} features', flush=True)
                if len(np.unique(y)) == 1:
                    predictions = np.full((len(factors), len(predict), HORIZON), float(y[0]), np.float32)
                    gain = np.zeros(len(names))
                else:
                    with threadpool_limits(limits=self.threads), threadpool_limits(limits=1, user_api='blas'):
                        model, _ = fit_member(self.specs[family], X[train_rows], y, max(counts), self.threads)
                        pred_X = X[pred_rows]
                        predictions = np.stack([staged_predict(model, self.specs[family]['backend'], pred_X, c, self.threads)
                                                .reshape(len(predict), HORIZON) for c in counts]).astype(np.float32)
                        gain = model.feature_importances_ if self.specs[family]['backend'] == 'xgboost' \
                            else model.feature_importance(importance_type='gain')
                    del model, pred_X
                pd.DataFrame(dict(feature=names, gain=gain)).sort_values('gain', ascending=False).to_csv(
                    folder / f'importance_{i:02d}.csv', index=False)
                temporary = checkpoint.with_suffix('.tmp.npz')
                np.savez_compressed(temporary, predictions=predictions, rounds=np.asarray(counts))
                os.replace(temporary, checkpoint)
                del X, y
                gc.collect()
                status = 'trained'
            if predictions.shape != (len(factors), len(predict), HORIZON) or not np.isfinite(predictions).all() \
                    or predictions.min() < 0 or predictions.max() > 1:
                raise ValueError(f'Invalid checkpoint: {checkpoint}')
            for k, f in enumerate(factors):
                cubes[factor_key(f)][:, i] = predictions[k]
            seconds = time.perf_counter() - started
            append_csv(self.directory / 'runtime_log.csv', [dict(
                stage=stage, family=family, variant=variant, unit=unit, status=status, seconds=round(seconds, 2),
                trees=max(counts), threads=self.threads, train_cases=len(train), signature=signature)])
            print(f'  {status}: {seconds:.1f}s ({i + 1}/{len(self.data.units)})', flush=True)
        return cubes


def group_diagnostics(data, cases, predictions, year):
    """Per unit/month/horizon/UTC-hour AUC and Brier, plus weekly-level MAE. Descriptive only."""
    truth = data.target_cube[cases].astype(np.float32)
    starts = pd.DatetimeIndex(data.train_cases.starttime.iloc[cases])
    hour_of_day = (starts.hour.to_numpy()[:, None] + np.arange(HORIZON)[None, :]) % 24
    blocks = [(0, 6), (6, 24), (24, 72), (72, 120), (120, 168)]
    rows = []

    def add(name, group, label, y, p, level=np.nan):
        auc = float(roc_auc_score(y.ravel(), p.ravel())) if 0 < y.mean() < 1 else np.nan
        rows.append(dict(year=year, model=name, group=group, label=label, auc=auc,
                         brier=float(np.mean((p - y) ** 2)), on_rate=float(y.mean()),
                         predicted_rate=float(p.mean()), weekly_level_mae=level))
    for name, p in predictions.items():
        add(name, 'all', 'all', truth, p, float(np.abs(p.mean(2) - truth.mean(2)).mean()))
        for i, unit in enumerate(data.units):
            add(name, 'unit', unit, truth[:, i], p[:, i], float(np.abs(p[:, i].mean(1) - truth[:, i].mean(1)).mean()))
        for month in range(1, 13):
            s = starts.month == month
            if s.any():
                add(name, 'start_month', month, truth[s], p[s])
        for a, b in blocks:
            add(name, 'horizon_hours', f'{a}-{b - 1}', truth[:, :, a:b], p[:, :, a:b])
        for h in range(0, 24, 3):
            m = np.broadcast_to(((hour_of_day >= h) & (hour_of_day < h + 3))[:, None, :], truth.shape)
            add(name, 'utc_hour', f'{h:02d}-{h + 2:02d}', truth[m], p[m])
    return pd.DataFrame(rows)


def block_bootstrap_delta(truth, candidate, control, block=7, reps=200, seed=0):
    """Paired bootstrap of AUC(candidate) - AUC(control), resampling blocks of consecutive cases
    (adjacent daily cases share 144 hours, so single cases are not independent)."""
    rng = np.random.default_rng(seed)
    n = len(truth)
    starts = np.arange(0, n, block)
    deltas = []
    for _ in range(reps):
        pick = np.concatenate([np.arange(s, min(s + block, n)) for s in rng.choice(starts, len(starts))])
        y = truth[pick].ravel()
        deltas.append(roc_auc_score(y, candidate[pick].ravel()) - roc_auc_score(y, control[pick].ravel()))
    return np.percentile(deltas, [5, 50, 95]), float(np.mean(np.asarray(deltas) > 0))


def evaluate_year(runner, year, stage):
    """Four fits x 14 units for one purged forward year; returns and saves all comparisons."""
    train, valid = purged_year(runner.data, year)
    truth = runner.data.target_cube[valid]
    started = time.perf_counter()
    single, rows = {}, []
    for family in FAMILIES:
        for variant in VARIANTS:
            t0 = time.perf_counter()
            snapshots = runner.run(family, variant, train, valid, f'{stage}_{year}')
            seconds = time.perf_counter() - t0
            for factor, p in snapshots.items():
                rows.append(dict(stage=stage, year=year, model=f'{family}_{variant}', factor=float(factor),
                                 micro_auc=score_cube(truth, p), seconds=round(seconds, 1)))
            single[(family, variant)] = snapshots[factor_key(DECISION_FACTOR)]
    blends = {v: XGB_WEIGHT * single[('xgb', v)] + (1 - XGB_WEIGHT) * single[('lgb', v)] for v in VARIANTS}
    for v, p in blends.items():
        rows.append(dict(stage=stage, year=year, model=f'blend_{v}', factor=DECISION_FACTOR,
                         micro_auc=score_cube(truth, p), seconds=round(time.perf_counter() - started, 1)))
    table = pd.DataFrame(rows)
    append_csv(runner.directory / 'results_log.csv', table.to_dict('records'))
    predictions = {'control_v5': blends['v5'], 'candidate_v5_lp': blends['v5_lp']}
    predictions.update({f'{f}_{v}': single[(f, v)] for f in FAMILIES for v in VARIANTS})
    diagnostics = group_diagnostics(runner.data, valid, predictions, year)
    diagnostics.to_csv(runner.directory / f'diagnostics_{stage}_{year}.csv', index=False)
    interval, share = block_bootstrap_delta(truth, blends['v5_lp'], blends['v5'])
    summary = dict(stage=stage, year=year, control=score_cube(truth, blends['v5']),
                   candidate=score_cube(truth, blends['v5_lp']),
                   delta_xgb=score_cube(truth, single[('xgb', 'v5_lp')]) - score_cube(truth, single[('xgb', 'v5')]),
                   delta_lgb=score_cube(truth, single[('lgb', 'v5_lp')]) - score_cube(truth, single[('lgb', 'v5')]),
                   bootstrap_delta_p05_p50_p95=[float(v) for v in interval], bootstrap_share_positive=share,
                   identity=runner.identity())
    summary['delta'] = summary['candidate'] - summary['control']
    save_json(runner.directory / f'summary_{stage}_{year}.json', summary)
    np.savez_compressed(runner.directory / f'predictions_{stage}_{year}.npz', case_ids=valid,
                        **{k: v.astype(np.float32) for k, v in predictions.items()})
    print(f'{year}: control v5 {summary["control"]:.6f} | candidate v5+LP {summary["candidate"]:.6f} | '
          f'delta {summary["delta"]:+.6f} (block bootstrap 90%: {interval[0]:+.6f} .. {interval[2]:+.6f})', flush=True)
    return summary, table, diagnostics


def select_candidate(runner, years):
    """Pre-registered rule, development years only: mean delta > 0 and no year worse than the tolerance."""
    summaries = []
    for year in years:
        path = runner.directory / f'summary_development_{year}.json'
        if not path.exists():
            raise FileNotFoundError(f'Run development year {year} first')
        s = json.loads(path.read_text())
        if s['identity'] != runner.identity():
            raise ValueError(f'Development {year} used another configuration; rerun it')
        summaries.append(s)
    deltas = [s['delta'] for s in summaries]
    accepted = bool(np.mean(deltas) > 0 and min(deltas) > -MAX_DEV_YEAR_LOSS)
    selection = dict(development_years=list(years), deltas=deltas, mean_delta=float(np.mean(deltas)),
                     accepted=accepted, variant='v5_lp' if accepted else 'v5', factor=DECISION_FACTOR,
                     xgb_weight=XGB_WEIGHT, rule=f'mean delta > 0 and every year > -{MAX_DEV_YEAR_LOSS}',
                     identity=runner.identity())
    save_json(runner.directory / 'selection.json', selection)
    return selection


def load_checked(runner, name):
    """A saved selection/summary only counts if this exact data/code/parameter setup produced it."""
    path = runner.directory / name
    if not path.exists():
        raise FileNotFoundError(f'{name} is missing; run the validation cells first')
    saved = json.loads(path.read_text())
    if saved.get('identity') != json.loads(json.dumps(runner.identity())):
        raise ValueError(f'{name} was produced by another data/code/parameter setup; rerun validation')
    return saved


def verdict(selection, audit):
    """Four outcomes; a positive point estimate whose 90% interval reaches zero is only 'uncertain'."""
    lo, _, hi = audit['bootstrap_delta_p05_p50_p95']
    if not selection['accepted']:
        return 'NOT SUPPORTED', 'Development rule rejected the LP features. Keep v5 selected; the CSV is experimental.'
    if audit['delta'] <= 0:
        return 'NOT CONFIRMED', 'Chosen on development years but not better on the later check. Keep v5 selected.'
    if lo <= 0:
        return 'POSITIVE BUT UNCERTAIN', (f'Point estimate {audit["delta"]:+.6f}, but the 90% interval {lo:+.6f} .. {hi:+.6f} '
                                          'includes no gain or a loss. An upload is a test, not evidence. Keep v5 selected.')
    return 'SUPPORTED', ('Accepted on development years and positive on the later check with the 90% interval above zero. '
                         'Upload hydro_lp_v6.csv as a new attempt; keep v5 as the fallback until the private score is known.')


def verify_submission(path, sample_path):
    submission, sample = pd.read_csv(path), pd.read_csv(sample_path)
    if list(submission.columns) != list(sample.columns) or submission.shape != sample.shape:
        raise AssertionError('Submission columns/shape differ from sample_submission.csv')
    if not submission['Run No'].equals(sample['Run No']):
        raise AssertionError('Run No order differs from sample_submission.csv')
    values = submission.iloc[:, 1:].to_numpy()
    if not np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise AssertionError('Invalid probabilities')
    return submission.shape


def final_lp(runner, variant='v5_lp', factor=DECISION_FACTOR, xgb_weight=XGB_WEIGHT, filename='hydro_lp_v6.csv'):
    """Fit the chosen pair on all labelled cases and write single-model and blend CSVs."""
    train = np.arange(runner.data.n_train_cases)
    test = np.arange(runner.data.n_train_cases, runner.data.n_all_cases)
    blend = np.zeros((len(test), len(runner.data.units), HORIZON), np.float32)
    paths = []
    for family in FAMILIES:
        p = runner.run(family, variant, train, test, 'final', factors=(factor,))[factor_key(factor)]
        blend += (xgb_weight if family == 'xgb' else 1 - xgb_weight) * p
        path, _ = make_submission(runner.data, p, f'{family}_{filename}')
        paths.append(path)
    path, submission = make_submission(runner.data, checked_cube(blend, len(test), len(runner.data.units)), filename)
    for p in paths + [path]:
        print('Verified', p, verify_submission(p, runner.data.root / 'sample_submission.csv'), flush=True)
    return path, submission

## Features og LP — ingen trening

In [11]:
# Input-only feature construction, including one LP per case. No estimator is fitted here.
data = build_project_data(ROOT)
extra, local, mapping = build_research_features(data)
additions, _ = build_refinement_features(data)
competition = additions['competition']
del additions  # the rejected competition+timing variant is not used
paths, path_audit = build_path_features(data, local)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
starts = pd.concat([data.train_cases, data.test_cases])['starttime']
lp_cache = ARTIFACTS / f'lp_cube_{SOURCE_DIGEST[:12]}_{raw_digest(ROOT)[:12]}.npz'
if USE_CHECKPOINTS and lp_cache.exists():
    with np.load(lp_cache) as saved:
        lp_cube, lp_plants, lp_seconds = saved['cube'], list(saved['plants']), float(saved['seconds'])
    print('Restored LP solutions:', lp_cache.name)
else:
    lp_cube, lp_plants, lp_seconds = solve_all_cases(ROOT, starts, THREADS)
    np.savez_compressed(lp_cache, cube=lp_cube, plants=np.array(lp_plants), seconds=lp_seconds)
print(f'LP relaxation: {len(lp_cube)} cases solved in {lp_seconds:.0f}s with {THREADS} threads')
n_cases = data.n_all_cases
price = data.X_all['price'].to_numpy().reshape(n_cases, HORIZON)
wv_source = {p: data.X_all[f'water_value_end__{PLANT_SOURCE[p]}'].to_numpy().reshape(n_cases, HORIZON)[:, 0]
             for p in lp_plants}
units_per_plant = {p: sum(u.rsplit('_G', 1)[0] == p for u in data.units) for p in lp_plants}
lp = lp_plant_features(lp_cube, lp_plants, price, wv_source, units_per_plant)
runner = LPRunner(data, extra, local, competition, paths, lp, SPECS, ROUNDS, THREADS,
                  SOURCE_DIGEST, ARTIFACTS, USE_CHECKPOINTS)
for variant in VARIANTS:
    print(variant, 'features per unit:', {p: len(runner.feature_names(p, variant)) for p in ('Tokke', 'Lio')})

LP relaxation: 3647 cases solved in 54s with 8 threads
v5 features per unit: {'Tokke': 334, 'Lio': 334}
v5_lp features per unit: {'Tokke': 347, 'Lio': 346}


## Fysikk-sjekk uten trening

In [12]:
# NO TRAINING. Physics sanity check: how well does ONE untrained score rank each unit's hours?
# rule = price - terminal local water value of the source reservoir; LP = reduced-cost margin.
# Labels are used descriptively only; nothing here feeds selection.
train_years = pd.DatetimeIndex(data.train_cases.starttime).year.to_numpy()
N = data.n_train_cases
rows = []
for i, unit in enumerate(data.units):
    plant = unit.rsplit('_G', 1)[0]
    j = lp_plants.index(plant)
    rule = (price - wv_source[plant][:, None])[:N]
    lp_score = lp_cube[:N, j, 2] + 1e-3 * (price[:N] - lp_cube[:N, j, 1])  # tie-break inside zero reduced cost
    for year in range(2015, 2023):
        s = train_years == year
        y = data.target_cube[s, i]
        rows.append(dict(unit=unit, year=year, rule_auc=roc_auc_score(y.ravel(), rule[s].ravel()),
                         lp_auc=roc_auc_score(y.ravel(), lp_score[s].ravel()),
                         lp_week_level_corr=np.corrcoef(lp_cube[:N][s, j, 0].mean(1), y.mean(1))[0, 1]))
physics = pd.DataFrame(rows)
physics.to_csv(ARTIFACTS / 'physics_check_untrained.csv', index=False)
display(physics.groupby('unit')[['rule_auc', 'lp_auc', 'lp_week_level_corr']].mean().round(4))

,rule_auc,lp_auc,lp_week_level_corr
unit,,,
Byrte_G1,0.9888,0.9946,0.9898
Haukeli_G1,0.7581,0.9514,0.8676
Hogga_G1,0.8991,0.9565,0.9540
Kjela_G1,0.9446,0.9771,0.9634
Lio_G1,0.8968,0.9616,0.7415
Songa_G1,0.9624,0.9848,0.9628
Tokke_G1,0.8943,0.9837,0.9241
Tokke_G2,0.9153,0.9853,0.9162
Tokke_G3,0.9112,0.9856,0.9701


## Endring i pris, vannverdi og LP-drift per år (ingen labels)

In [13]:
# NO TRAINING, NO LABELS: input distribution by case-start year, including the 2023-2024 test years.
years = pd.DatetimeIndex(starts).year.to_numpy()
rows = []
for year in sorted(set(years)):
    s = years == year
    row = dict(year=year, price_mean=price[s].mean(), price_std_within_week=price[s].std(axis=1).mean())
    for p in ('Tokke', 'Vinje', 'Lio', 'Haukeli', 'Hogga'):
        j = lp_plants.index(p)
        row[f'{p}_wv_mean'] = wv_source[p][s].mean()
        row[f'{p}_lp_week_fraction'] = lp_cube[s, j, 0].mean()
        # Hours the LP runs the plant although price is below its own water value (coupling-forced).
        row[f'{p}_lp_forced_share'] = ((lp_cube[s, j, 0] > .01) & (price[s] < wv_source[p][s][:, None])).mean()
    rows.append(row)
shift_table = pd.DataFrame(rows).set_index('year')
shift_table.to_csv(ARTIFACTS / 'input_shift_by_year.csv')
display(shift_table.round(3))

,price_mean,price_std_within_week,Tokke_wv_mean,Tokke_lp_week_fraction,Tokke_lp_forced_share,Vinje_wv_mean,Vinje_lp_week_fraction,Vinje_lp_forced_share,Lio_wv_mean,Lio_lp_week_fraction,Lio_lp_forced_share,Haukeli_wv_mean,Haukeli_lp_week_fraction,Haukeli_lp_forced_share,Hogga_wv_mean,Hogga_lp_week_fraction,Hogga_lp_forced_share
year,,,,,,,,,,,,,,,,,
2015,19.732000,2.893000,18.750000,0.630,0.135,18.312000,0.361,0.087,18.417000,0.630,0.244,3.727000,0.833,0.037,20.593000,0.590,0.124
2016,25.259001,2.539000,22.944000,0.629,0.150,26.341999,0.406,0.098,29.999001,0.526,0.347,16.020000,0.740,0.105,29.184999,0.539,0.136
2017,28.823999,2.736000,27.757999,0.580,0.123,28.250000,0.362,0.088,10.403000,0.928,0.082,17.188000,0.690,0.102,35.291000,0.534,0.170
2018,43.433998,4.902000,37.146000,0.686,0.140,37.254002,0.462,0.098,39.784000,0.666,0.207,23.500999,0.688,0.036,40.784000,0.696,0.105
2019,39.089001,3.245000,40.917999,0.482,0.110,40.219002,0.302,0.101,32.012001,0.765,0.295,34.884998,0.402,0.066,43.987000,0.578,0.102
2020,9.298000,2.074000,8.503000,0.687,0.177,6.637000,0.432,0.015,7.952000,0.669,0.309,12.609000,0.163,0.080,6.207000,0.767,0.065
2021,75.927002,15.995000,84.192001,0.553,0.125,58.789001,0.395,0.052,114.379997,0.419,0.407,98.832001,0.288,0.075,87.773003,0.567,0.097
2022,211.240997,49.627998,245.755005,0.442,0.045,281.699005,0.294,0.073,303.108002,0.414,0.483,372.615997,0.084,0.047,244.804001,0.413,0.066
2023,78.947998,22.131001,79.287003,0.469,0.041,118.643997,0.258,0.106,100.529999,0.515,0.499,131.121002,0.252,0.120,81.814003,0.515,0.045


## TRENING A — modellvalg (2020, 2021)

In [14]:
# TRAINING A - model selection years. 4 fits x 14 units per year; completed units resume.
for year in DEV_YEARS:
    evaluate_year(runner, year, 'development')
log = pd.read_csv(ARTIFACTS / 'results_log.csv').drop_duplicates(['stage', 'year', 'model', 'factor'], keep='last')
display(log.query("stage == 'development'").pivot_table(index=['model', 'factor'], columns='year', values='micro_auc').round(6))

development_2020 | xgb/v5 | Hogga_G1 | 820 trees, 334 features
  trained: 26.0s (1/14)
development_2020 | xgb/v5 | Tokke_G1 | 900 trees, 334 features
  trained: 32.8s (2/14)
development_2020 | xgb/v5 | Tokke_G2 | 1708 trees, 334 features
  trained: 55.3s (3/14)
development_2020 | xgb/v5 | Tokke_G3 | 1174 trees, 334 features
  trained: 39.1s (4/14)
development_2020 | xgb/v5 | Tokke_G4 | 1386 trees, 334 features
  trained: 43.5s (5/14)
development_2020 | xgb/v5 | Lio_G1 | 498 trees, 334 features
  trained: 15.8s (6/14)
development_2020 | xgb/v5 | Byrte_G1 | 584 trees, 334 features
  trained: 18.7s (7/14)
development_2020 | xgb/v5 | Vinje_G1 | 1316 trees, 334 features
  trained: 42.3s (8/14)
development_2020 | xgb/v5 | Vinje_G2 | 808 trees, 334 features
  trained: 26.7s (9/14)
development_2020 | xgb/v5 | Vinje_G3 | 1214 trees, 334 features
  trained: 37.7s (10/14)
development_2020 | xgb/v5 | Haukeli_G1 | 446 trees, 334 features
  trained: 15.4s (11/14)
development_2020 | xgb/v5 | Songa_G1

year                    2020      2021
model       factor                    
blend_v5    2.0     0.966790  0.985145
blend_v5_lp 2.0     0.987367  0.993805
lgb_v5      1.0     0.965162  0.984376
            1.5     0.965503  0.984738
            2.0     0.965375  0.984838
lgb_v5_lp   1.0     0.986749  0.993434
            1.5     0.987050  0.993606
            2.0     0.987149  0.993659
xgb_v5      1.0     0.968333  0.984467
            1.5     0.967773  0.984685
            2.0     0.967154  0.984750
xgb_v5_lp   1.0     0.987079  0.993390
            1.5     0.987147  0.993543
            2.0     0.987141  0.993630

## Valg etter forhåndsregistrert regel — ingen trening

In [17]:
TRAIN_FINAL = True
LOCAL_CPU_THREADS = 12 

In [18]:
# NO TRAINING: apply the pre-registered rule to the development years only.
selection = select_candidate(runner, DEV_YEARS)
print(json.dumps({k: v for k, v in selection.items() if k != 'identity'}, indent=1))
units = pd.concat([pd.read_csv(ARTIFACTS / f'diagnostics_development_{y}.csv') for y in DEV_YEARS])
display(units.query("group == 'unit' and model in ['control_v5', 'candidate_v5_lp']")
        .pivot_table(index='label', columns=['year', 'model'], values=['auc', 'weekly_level_mae']).round(4))

{
 "development_years": [
  2020,
  2021
 ],
 "deltas": [
  0.02057730668621527,
  0.008660193188004484
 ],
 "mean_delta": 0.014618749937109876,
 "accepted": true,
 "variant": "v5_lp",
 "factor": 2.0,
 "xgb_weight": 0.5,
 "rule": "mean delta > 0 and every year > -0.0005"
}


auc                                        \
year                      2020                       2021              
model          candidate_v5_lp control_v5 candidate_v5_lp control_v5   
label                                                                  
Byrte_G1                0.9988     0.9960          0.9994     0.9993   
Haukeli_G1              0.9657     0.9077          0.9745     0.9452   
Hogga_G1                0.9864     0.9781          0.9792     0.9753   
Kjela_G1                0.9946     0.9902          0.9921     0.9875   
Lio_G1                  0.9922     0.9796          0.9873     0.9661   
Songa_G1                0.9926     0.9824          0.9991     0.9981   
Tokke_G1                0.9904     0.9815          0.9970     0.9897   
Tokke_G2                0.9868     0.9659          0.9959     0.9876   
Tokke_G3                0.9918     0.9825          0.9968     0.9894   
Tokke_G4                0.9858     0.9684          0.9949     0.9830   
Vesle Kjela_G1          0.9923     0.9884          0.9995     0.9995   
Vinje_G1                0.9888     0.9686          0.9941     0.9806   
Vinje_G2                0.9799     0.9649          0.9888     0.9752   
Vinje_G3                0.9521     0.9348          0.9830     0.9765   

               weekly_level_mae                                        
year                       2020                       2021             
model           candidate_v5_lp control_v5 candidate_v5_lp control_v5  
label                                                                  
Byrte_G1                 0.0173     0.0277          0.0120     0.0131  
Haukeli_G1               0.1503     0.3858          0.0733     0.1672  
Hogga_G1                 0.0385     0.0595          0.0600     0.0696  
Kjela_G1                 0.0401     0.0622          0.0202     0.0342  
Lio_G1                   0.0377     0.0668          0.0626     0.1160  
Songa_G1                 0.0433     0.0664          0.0192     0.0239  
Tokke_G1                 0.0366     0.0537          0.0248     0.0504  
Tokke_G2                 0.0659     0.1076          0.0318     0.0632  
Tokke_G3                 0.0442     0.0752          0.0238     0.0518  
Tokke_G4                 0.0708     0.0974          0.0351     0.0733  
Vesle Kjela_G1           0.0498     0.0552          0.0124     0.0137  
Vinje_G1                 0.0520     0.0862          0.0315     0.0648  
Vinje_G2                 0.0781     0.1083          0.0414     0.0680  
Vinje_G3                 0.0982     0.1169          0.0452     0.0583

## TRENING B — etterfølgende kontroll (2022)

In [19]:
# TRAINING B - later check on AUDIT_YEAR, run once after selection. Do not retune on it.
selection = json.loads((ARTIFACTS / 'selection.json').read_text())
if selection['identity'] != runner.identity():
    raise ValueError('Configuration changed after selection; rerun development')
audit_summary, audit_table, audit_diag = evaluate_year(runner, AUDIT_YEAR, 'audit')
display(audit_table.pivot_table(index='model', columns='factor', values='micro_auc').round(6))
for group in ('unit', 'start_month', 'horizon_hours', 'utc_hour'):
    display(audit_diag.query("group == @group and model in ['control_v5', 'candidate_v5_lp']")
            .pivot_table(index='label', columns='model', values=['auc', 'weekly_level_mae'] if group == 'unit' else 'auc').round(4))
old = ROOT / 'artifacts/multiscale_boosting_v5/audit_predictions.npz'
if old.exists():  # consistency check: our control should reproduce notebook 18's 2022 candidate
    with np.load(old) as saved:
        mine = np.load(ARTIFACTS / f'predictions_audit_{AUDIT_YEAR}.npz')
        if AUDIT_YEAR == 2022 and np.array_equal(saved['case_ids'], mine['case_ids']):
            print('max |control - notebook18 v5| =', float(np.abs(saved['candidate'] - mine['control_v5']).max()))

  restored: 0.0s (1/14)
  restored: 0.0s (2/14)
  restored: 0.0s (3/14)
audit_2022 | xgb/v5 | Tokke_G3 | 1174 trees, 334 features
  trained: 55.5s (4/14)
audit_2022 | xgb/v5 | Tokke_G4 | 1386 trees, 334 features
  trained: 61.6s (5/14)
audit_2022 | xgb/v5 | Lio_G1 | 498 trees, 334 features
  trained: 22.8s (6/14)
audit_2022 | xgb/v5 | Byrte_G1 | 584 trees, 334 features
  trained: 26.2s (7/14)
audit_2022 | xgb/v5 | Vinje_G1 | 1316 trees, 334 features
  trained: 60.5s (8/14)
audit_2022 | xgb/v5 | Vinje_G2 | 808 trees, 334 features
  trained: 38.8s (9/14)
audit_2022 | xgb/v5 | Vinje_G3 | 1214 trees, 334 features
  trained: 53.5s (10/14)
audit_2022 | xgb/v5 | Haukeli_G1 | 446 trees, 334 features
  trained: 21.4s (11/14)
audit_2022 | xgb/v5 | Songa_G1 | 1020 trees, 334 features
  trained: 50.0s (12/14)
audit_2022 | xgb/v5 | Kjela_G1 | 652 trees, 334 features
  trained: 31.6s (13/14)
audit_2022 | xgb/v5 | Vesle Kjela_G1 | 744 trees, 334 features
  trained: 33.5s (14/14)
audit_2022 | xgb/v5_l

factor,1.0,1.5,2.0
model,,,
blend_v5,NaN,NaN,0.986819
blend_v5_lp,NaN,NaN,0.992564
lgb_v5,0.986088,0.986301,0.986496
lgb_v5_lp,0.991884,0.992291,0.992425
xgb_v5,0.985679,0.986321,0.986579
xgb_v5_lp,0.992117,0.992331,0.992437


auc            weekly_level_mae           
model          candidate_v5_lp control_v5  candidate_v5_lp control_v5
label                                                                
Byrte_G1                0.9995     0.9987           0.0075     0.0163
Haukeli_G1              0.9403     0.9463           0.0477     0.0848
Hogga_G1                0.9838     0.9788           0.0510     0.0640
Kjela_G1                0.9879     0.9876           0.0383     0.0497
Lio_G1                  0.9757     0.9626           0.0794     0.1135
Songa_G1                0.9959     0.9960           0.0092     0.0106
Tokke_G1                0.9980     0.9948           0.0176     0.0375
Tokke_G2                0.9963     0.9885           0.0274     0.0559
Tokke_G3                0.9980     0.9930           0.0197     0.0434
Tokke_G4                0.9969     0.9883           0.0256     0.0585
Vesle Kjela_G1          0.9957     0.9918           0.0279     0.0293
Vinje_G1                0.9935     0.9864           0.0362     0.0500
Vinje_G2                0.9899     0.9820           0.0357     0.0512
Vinje_G3                0.9820     0.9744           0.0393     0.0510

model,candidate_v5_lp,control_v5
label,,
1,0.9938,0.9909
2,0.9969,0.9945
3,0.9956,0.9932
4,0.9614,0.9628
5,0.9908,0.9831
6,0.9856,0.9755
7,0.9825,0.9745
8,0.9973,0.9961
9,0.9978,0.9936


model,candidate_v5_lp,control_v5
label,,
0-5,0.9910,0.9827
120-167,0.9936,0.9887
24-71,0.9923,0.9862
6-23,0.9930,0.9851
72-119,0.9918,0.9866


model,candidate_v5_lp,control_v5
label,,
00-02,0.9904,0.9842
03-05,0.9922,0.9862
06-08,0.9932,0.9869
09-11,0.9928,0.9872
12-14,0.9928,0.9879
15-17,0.9932,0.9873
18-20,0.9932,0.9874
21-23,0.9918,0.9860


max |control - notebook18 v5| = 0.0


## Konklusjon før slutttrening — ingen trening

In [20]:
# NO TRAINING: verdict from the saved summaries, only if they match this exact setup.
selection = load_checked(runner, 'selection.json')
audit_summary = load_checked(runner, f'summary_audit_{AUDIT_YEAR}.json')
lo, mid, hi = audit_summary['bootstrap_delta_p05_p50_p95']
print('Development deltas:', dict(zip(selection['development_years'], np.round(selection['deltas'], 6))),
      '| rule:', selection['rule'], '| accepted:', selection['accepted'])
print(f"{AUDIT_YEAR}: control {audit_summary['control']:.6f}, candidate {audit_summary['candidate']:.6f}, "
      f"delta {audit_summary['delta']:+.6f}, 90% block bootstrap {lo:+.6f} .. {hi:+.6f}, "
      f"share of resamples > 0: {audit_summary['bootstrap_share_positive']:.2f}")
label, message = verdict(selection, audit_summary)
print(label + ':', message)
print('2020-2022 all informed earlier work, and 2022 informed this hypothesis: none is an untouched holdout.')

Development deltas: {2020: np.float64(0.020577), 2021: np.float64(0.00866)} | rule: mean delta > 0 and every year > -0.0005 | accepted: True
2022: control 0.986819, candidate 0.992564, delta +0.005744, 90% block bootstrap +0.003690 .. +0.008593, share of resamples > 0: 1.00
SUPPORTED: Accepted on development years and positive on the later check with the 90% interval above zero. Upload hydro_lp_v6.csv as a new attempt; keep v5 as the fallback until the private score is known.
2020-2022 all informed earlier work, and 2022 informed this hypothesis: none is an untouched holdout.


## TRENING C — endelig Kaggle-CSV (sjekker samme oppsett som valideringen)

In [21]:
# TRAINING C - candidate pair on all labelled cases (2 x 14 fits) and the Kaggle CSVs.
# Refuses to run unless selection and later check were produced by this exact setup.
selection = load_checked(runner, 'selection.json')
audit_summary = load_checked(runner, f'summary_audit_{AUDIT_YEAR}.json')
label, message = verdict(selection, audit_summary)
print('Verdict for this setup:', label)
if TRAIN_FINAL:
    submission_path, submission = final_lp(runner)
    save_json(ARTIFACTS / 'final_manifest.json', dict(
        csv=str(submission_path.relative_to(ROOT)), verdict=label, message=message,
        development_deltas=selection['deltas'], audit_delta=audit_summary['delta'],
        audit_interval_p05_p50_p95=audit_summary['bootstrap_delta_p05_p50_p95'], identity=runner.identity()))
    display(submission.iloc[:3, :6])
else:
    print('TRAIN_FINAL = False: stopped before the final fits.')
print('Total minutes in this kernel:', round((time.perf_counter() - NOTEBOOK_STARTED) / 60, 1))

Verdict for this setup: SUPPORTED
final | xgb/v5_lp | Hogga_G1 | 820 trees, 346 features
  trained: 39.9s (1/14)
final | xgb/v5_lp | Tokke_G1 | 900 trees, 347 features
  trained: 49.6s (2/14)
final | xgb/v5_lp | Tokke_G2 | 1708 trees, 347 features
  trained: 86.1s (3/14)
final | xgb/v5_lp | Tokke_G3 | 1174 trees, 347 features
  trained: 61.0s (4/14)
final | xgb/v5_lp | Tokke_G4 | 1386 trees, 347 features
  trained: 68.8s (5/14)
final | xgb/v5_lp | Lio_G1 | 498 trees, 346 features
  trained: 25.1s (6/14)
final | xgb/v5_lp | Byrte_G1 | 584 trees, 346 features
  trained: 28.6s (7/14)
final | xgb/v5_lp | Vinje_G1 | 1316 trees, 347 features
  trained: 68.6s (8/14)
final | xgb/v5_lp | Vinje_G2 | 808 trees, 347 features
  trained: 42.4s (9/14)
final | xgb/v5_lp | Vinje_G3 | 1214 trees, 347 features
  trained: 62.9s (10/14)
final | xgb/v5_lp | Haukeli_G1 | 446 trees, 346 features
  trained: 24.2s (11/14)
final | xgb/v5_lp | Songa_G1 | 1020 trees, 346 features
  trained: 55.2s (12/14)
final | x

,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4
0,2923,0.059583,0.026982,0.007063,0.003769,0.004645
1,2924,0.757048,0.756073,0.765026,0.782472,0.800023
2,2925,0.999679,0.999703,0.999751,0.999775,0.999709


Total minutes in this kernel: 158.9


## Filer og hva du gjør etterpå

- `submissions/hydro_lp_v6.csv` (blend), `submissions/xgb_hydro_lp_v6.csv`, `submissions/lgb_hydro_lp_v6.csv`.
- `artifacts/hydro_lp_v6/`: `results_log.csv`, `runtime_log.csv`, `summary_*.json`, `diagnostics_*.csv`,
  `predictions_*.npz`, `selection.json`, `physics_check_untrained.csv`, `input_shift_by_year.csv`,
  importanser per enhet, LP-cache.
- Professor-notebook: `artifacts/hydro_lp_v6/Short_notebook_hydro_lp_v6.ipynb` (bygd av
  `scripts/build_hydro_lp_notebook.py`; selvstendig, 4 tråder, 28 fits, ingen cache).

`submissions/experiment_log.csv` endres ikke automatisk; legg inn en rad selv etter opplasting.
Last opp `hydro_lp_v6.csv` bare som et nytt forsøk; behold v5 som valgt fallback til private-
scoren er kjent. Offentlig leaderboard er én delmengde; bruk den varsomt.

Kilder: [SHOP reservoar](https://docs.shop.sintef.energy/objects/reservoir/reservoir.html),
[SHOP vannverdier](https://docs.shop.sintef.energy/examples/water_values/water_values.html),
[SHOP elv](https://docs.shop.sintef.energy/objects/river/river.html),
[SINTEF InterOpt](https://www.sintef.no/en/latest-news/2026/how-ai-could-squeeze-more-value-from-norwegian-hydropower/),
[Xavier, Qiu & Ahmed 2021](https://pubsonline.informs.org/doi/10.1287/ijoc.2020.0976),
[Gasse et al. 2019](https://arxiv.org/abs/1906.01629),
[SciPy HiGHS](https://docs.scipy.org/doc/scipy/reference/optimize.linprog-highs.html),
[scikit-learn TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html).
Detaljer: `RESEARCH_HYDRO_LP_V6.md`.